# 株・為替 取引シミュレーター

yfinance（Yahoo Finance）の価格を使って、**キオクシア・レーザーテック・ドル円・ユーロ円**のロング（買い）とショート（売り）を仮想的に取引し、損益を記録します。

## 使い方

1. メニューの **ランタイム → すべてのセルを実行** を選びます（または上から順に ▶ を押します）。
2. Google ドライブへのアクセスを求められたら許可してください。取引データはドライブの `trade_sim/portfolio.json` に保存され、次に開いたときも続きから使えます。
3. ②のセルの下に表示される画面で、銘柄・売買・数量を選んで **発注する** を押します。決済は「保有中の建玉」で行います。

- 約定日（YYYY-MM-DD）を入れると、その日の終値で約定します（過去の価格でのシミュレーション）。空欄なら最新の価格です。
- 価格は Yahoo Finance のデータです。東証の銘柄はおよそ 20 分遅れです。
- 手数料、スプレッド、スワップポイント、信用取引の金利・貸株料は計算に含めていません。

In [ ]:
#@title ① セットアップ（最初に実行してください） { display-mode: "form" }
#@markdown **save_to_drive**: チェックを入れておくと取引データを Google ドライブに保存し、次に開いたときも続きから使えます。
save_to_drive = True  #@param {type:"boolean"}

import importlib.metadata
import re
import subprocess
import sys
from pathlib import Path


def _version(package):
    try:
        return tuple(int(x) for x in re.findall(r"\d+", importlib.metadata.version(package))[:3])
    except importlib.metadata.PackageNotFoundError:
        return ()


# yfinance が入っていないか古ければ、最新版を入れる
if _version("yfinance") < (1, 7, 0):
    print("yfinance をインストールしています…")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "yfinance"], check=True)

# シミュレーター本体 (trade_sim パッケージ) を書き出す
_SOURCES = {
    'trade_sim/__init__.py': r'''"""yfinance の価格を使った株・為替の取引シミュレーター。"""

__version__ = "0.1.0"
''',
    'trade_sim/__main__.py': r'''from .cli import main

raise SystemExit(main())
''',
    'trade_sim/charts.py': r'''"""チャート (Plotly) を作る。

色は日本の証券会社の表示に合わせて、値上がり・プラスを赤、値下がり・マイナスを青にしている。
"""

from __future__ import annotations

from collections.abc import Sequence
from datetime import datetime, timedelta

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from .instruments import Instrument, get_instrument
from .portfolio import LONG, SIDE_LABELS, Position, SummaryRow, Trade, parse_trade_time
from .prices import Timeframe
from .simulator import fmt_yen

UP = "#d9434f"  # 陽線・プラス
DOWN = "#2f6bd6"  # 陰線・マイナス
EXIT = "#8a8f98"  # 決済の印
MA_COLORS = ("#e8890c", "#2a9d63", "#9549c4")
GRID = "rgba(127, 127, 127, 0.18)"


trade_time = parse_trade_time  # 記録された約定日時を datetime にする


def locate(index: pd.DatetimeIndex, when: datetime) -> int | None:
    """when を含むローソク足の位置。チャートの最初の足より前なら None。"""
    position = int(index.searchsorted(pd.Timestamp(when), side="right")) - 1
    return position if position >= 0 else None


_PERIOD_OFFSETS = {
    "1mo": pd.DateOffset(months=1),
    "3mo": pd.DateOffset(months=3),
    "6mo": pd.DateOffset(months=6),
    "1y": pd.DateOffset(years=1),
    "2y": pd.DateOffset(years=2),
    "5y": pd.DateOffset(years=5),
    "10y": pd.DateOffset(years=10),
}


def visible_start(index: pd.DatetimeIndex, period: str) -> int:
    """表示する期間 (period) の最初の足の位置。それより前の足は移動平均線の計算にだけ使う。"""
    if len(index) == 0 or period == "max":
        return 0
    if period in ("1d", "5d"):
        days = sorted(set(index.normalize()))
        count = 1 if period == "1d" else 5
        cutoff = days[-count] if len(days) >= count else days[0]
    else:
        cutoff = index[-1] - _PERIOD_OFFSETS[period]
    return int(index.searchsorted(cutoff, side="left"))


def _labels(index: pd.DatetimeIndex, timeframe: Timeframe) -> list[str]:
    # 夜間や休日のすき間ができないように、横軸は時刻ではなく足の並び (カテゴリ) にする
    fmt = "%m/%d %H:%M" if timeframe.intraday else "%Y/%m/%d"
    return [ts.strftime(fmt) for ts in index]


def _ma_name(window: int, timeframe: Timeframe) -> str:
    if timeframe.key == "1d":
        return f"{window}日線"
    if timeframe.key == "1wk":
        return f"{window}週線"
    return f"移動平均({window})"


def candle_figure(
    inst: Instrument,
    candles: pd.DataFrame,
    timeframe: Timeframe,
    period: str,
    trades: Sequence[Trade] = (),
    positions: Sequence[Position] = (),
    show_ma: bool = True,
    show_volume: bool = True,
) -> go.Figure:
    """ローソク足チャート。移動平均線、出来高、自分の売買の位置も描く。

    candles には表示する期間より前の足も含めてよい (移動平均線の計算にだけ使う)。
    """
    start = visible_start(candles.index, period)
    averages = {window: candles["Close"].rolling(window).mean().iloc[start:] for window in timeframe.ma_windows}
    candles = candles.iloc[start:]
    labels = _labels(candles.index, timeframe)
    has_volume = show_volume and inst.kind == "stock" and float(candles["Volume"].sum()) > 0
    if has_volume:
        fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.78, 0.22], vertical_spacing=0.03)
    else:
        fig = make_subplots(rows=1, cols=1)

    fig.add_trace(
        go.Candlestick(
            x=labels,
            open=candles["Open"],
            high=candles["High"],
            low=candles["Low"],
            close=candles["Close"],
            name=inst.name,
            increasing={"line": {"color": UP, "width": 1}, "fillcolor": UP},
            decreasing={"line": {"color": DOWN, "width": 1}, "fillcolor": DOWN},
        ),
        row=1,
        col=1,
    )

    if show_ma:
        for window, color in zip(timeframe.ma_windows, MA_COLORS):
            average = averages[window]
            if average.notna().any():
                fig.add_trace(
                    go.Scatter(
                        x=labels,
                        y=average,
                        mode="lines",
                        name=_ma_name(window, timeframe),
                        line={"color": color, "width": 1.4},
                        hovertemplate=f"%{{y:,.{inst.price_decimals}f}}",
                    ),
                    row=1,
                    col=1,
                )

    if has_volume:
        colors = [UP if c >= o else DOWN for o, c in zip(candles["Open"], candles["Close"])]
        fig.add_trace(
            go.Bar(x=labels, y=candles["Volume"], name="出来高", marker={"color": colors, "opacity": 0.55}),
            row=2,
            col=1,
        )

    _add_trade_marks(fig, inst, candles, labels, trades, positions)

    price_format = f",.{inst.price_decimals}f"
    fig.update_layout(
        height=600 if has_volume else 520,
        margin={"l": 8, "r": 8, "t": 36, "b": 8},
        hovermode="x unified",
        legend={"orientation": "h", "yanchor": "bottom", "y": 1.01, "xanchor": "left", "x": 0},
        xaxis_rangeslider_visible=False,
        dragmode="pan",
    )
    fig.update_xaxes(type="category", nticks=8, showgrid=True, gridcolor=GRID, showspikes=True, spikethickness=1)
    fig.update_yaxes(showgrid=True, gridcolor=GRID, tickformat=price_format, row=1, col=1)
    if has_volume:
        fig.update_yaxes(showgrid=True, gridcolor=GRID, tickformat="~s", row=2, col=1)
    return fig


def _add_trade_marks(
    fig: go.Figure,
    inst: Instrument,
    candles: pd.DataFrame,
    labels: list[str],
    trades: Sequence[Trade],
    positions: Sequence[Position],
) -> None:
    """建てた位置 (▲ロング / ▼ショート) と決済した位置 (×) を描き、保有中の建値に線を引く。"""
    index = candles.index
    entries: dict[int, tuple[str, str, float, int]] = {}  # 建玉 ID -> (side, opened_at, 建値, 数量)
    exits_x, exits_y, exits_text = [], [], []
    for trade in trades:
        if trade.symbol != inst.key:
            continue
        entries.setdefault(trade.position_id, (trade.side, trade.opened_at, trade.entry_price, 0))
        side, opened_at, entry, qty = entries[trade.position_id]
        entries[trade.position_id] = (side, opened_at, entry, qty + trade.qty)
        i = locate(index, trade_time(trade.closed_at))
        if i is not None:
            exits_x.append(labels[i])
            exits_y.append(trade.exit_price)
            exits_text.append(
                f"決済 #{trade.position_id} {SIDE_LABELS[trade.side]} {inst.format_qty(trade.qty)}"
                f" @ {inst.format_price(trade.exit_price)}　損益 {fmt_yen(trade.pnl)}"
            )
    for position in positions:
        if position.symbol != inst.key:
            continue
        side, opened_at, entry, qty = entries.get(
            position.id, (position.side, position.opened_at, position.entry_price, 0)
        )
        entries[position.id] = (side, opened_at, entry, qty + position.qty)
        color = UP if position.side == LONG else DOWN
        fig.add_hline(
            y=position.entry_price,
            line={"color": color, "width": 1, "dash": "dot"},
            annotation_text=f"#{position.id} {SIDE_LABELS[position.side]} 建値 {inst.format_price(position.entry_price)}",
            annotation_position="top right",
            annotation_font={"color": color, "size": 11},
            annotation_bgcolor="rgba(127, 127, 127, 0.12)",
            row=1,
            col=1,
        )
        for price, label, line_color in ((position.take_profit, "利確", UP), (position.stop_loss, "損切り", DOWN)):
            if price is None:
                continue
            fig.add_hline(
                y=price,
                line={"color": line_color, "width": 1, "dash": "dash"},
                annotation_text=f"#{position.id} {label} {inst.format_price(price)}",
                annotation_position="bottom right",
                annotation_font={"color": line_color, "size": 11},
                annotation_bgcolor="rgba(127, 127, 127, 0.12)",
                row=1,
                col=1,
            )

    for side, symbol, color, name in ((LONG, "triangle-up", UP, "ロング"), ("short", "triangle-down", DOWN, "ショート")):
        xs, ys, texts = [], [], []
        for position_id, (entry_side, opened_at, entry, qty) in sorted(entries.items()):
            if entry_side != side:
                continue
            i = locate(index, trade_time(opened_at))
            if i is None:
                continue
            xs.append(labels[i])
            ys.append(entry)
            texts.append(f"#{position_id} {name} {inst.format_qty(qty)} @ {inst.format_price(entry)}")
        if xs:
            fig.add_trace(
                go.Scatter(
                    x=xs,
                    y=ys,
                    mode="markers",
                    name=f"{name}で建てた位置",
                    marker={"symbol": symbol, "size": 13, "color": color, "line": {"width": 1, "color": "white"}},
                    text=texts,
                    hovertemplate="%{text}<extra></extra>",
                ),
                row=1,
                col=1,
            )
    if exits_x:
        fig.add_trace(
            go.Scatter(
                x=exits_x,
                y=exits_y,
                mode="markers",
                name="決済した位置",
                marker={"symbol": "x", "size": 11, "color": EXIT, "line": {"width": 1, "color": "white"}},
                text=exits_text,
                hovertemplate="%{text}<extra></extra>",
            ),
            row=1,
            col=1,
        )


def realized_curve(trades: Sequence[Trade]) -> pd.DataFrame:
    """決済ごとの実現損益と、その累計 (時刻順)。"""
    rows = sorted(((trade_time(t.closed_at), t) for t in trades), key=lambda item: item[0])
    data = {
        "time": [when for when, _ in rows],
        "pnl": [t.pnl for _, t in rows],
        "label": [f"#{t.position_id} {get_instrument(t.symbol).name} {SIDE_LABELS[t.side]}" for _, t in rows],
    }
    df = pd.DataFrame(data)
    df["cumulative"] = df["pnl"].cumsum().round(2) if not df.empty else pd.Series(dtype=float)
    return df


def equity_figure(trades: Sequence[Trade], current_total: float | None, now: datetime) -> go.Figure:
    """通算損益の推移。決済のたびに実現損益の累計が階段状に変わり、最後に評価損益を含む現在の通算損益を置く。"""
    curve = realized_curve(trades)
    fig = go.Figure()
    if not curve.empty:
        start = curve["time"].iloc[0] - timedelta(minutes=1)
        xs = [start, *curve["time"]]
        ys = [0.0, *curve["cumulative"]]
        texts = ["開始", *[f"{label}　{fmt_yen(pnl)}" for label, pnl in zip(curve["label"], curve["pnl"])]]
        fig.add_trace(
            go.Scatter(
                x=xs,
                y=ys,
                mode="lines+markers",
                line={"shape": "hv", "color": "#5b6b80", "width": 2},
                marker={"size": 7, "color": ["#5b6b80", *[UP if v > 0 else DOWN for v in curve["pnl"]]]},
                name="実現損益の累計",
                text=texts,
                hovertemplate="%{x|%Y/%m/%d %H:%M}<br>%{text}<br>累計 %{y:+,.0f}円<extra></extra>",
            )
        )
    if current_total is not None:
        fig.add_trace(
            go.Scatter(
                x=[now],
                y=[current_total],
                mode="markers",
                marker={"symbol": "star", "size": 15, "color": UP if current_total >= 0 else DOWN},
                name="現在 (評価損益を含む)",
                hovertemplate="現在の通算損益 %{y:+,.0f}円<extra></extra>",
            )
        )
    fig.add_hline(y=0, line={"color": EXIT, "width": 1})
    fig.update_layout(
        height=380,
        margin={"l": 8, "r": 8, "t": 36, "b": 8},
        legend={"orientation": "h", "yanchor": "bottom", "y": 1.01, "xanchor": "left", "x": 0},
        yaxis={"tickformat": "+,.0f", "ticksuffix": "円", "gridcolor": GRID},
        xaxis={"gridcolor": GRID, "tickformat": "%Y/%m/%d"},
    )
    return fig


def pnl_by_instrument_figure(rows: Sequence[SummaryRow]) -> go.Figure:
    """銘柄別の損益 (実現損益 + 評価損益)。現在値を取得できなかった銘柄は実現損益だけ。"""
    names = [get_instrument(r.symbol).name for r in rows]
    values = [r.total if r.total is not None else r.realized for r in rows]
    fig = go.Figure(
        go.Bar(
            x=names,
            y=values,
            marker={"color": [UP if v >= 0 else DOWN for v in values]},
            text=[fmt_yen(v) for v in values],
            textposition="outside",
            hovertemplate="%{x}<br>%{y:+,.0f}円<extra></extra>",
            cliponaxis=False,
        )
    )
    fig.add_hline(y=0, line={"color": EXIT, "width": 1})
    fig.update_layout(
        height=380,
        margin={"l": 8, "r": 8, "t": 36, "b": 8},
        showlegend=False,
        yaxis={"tickformat": "+,.0f", "ticksuffix": "円", "gridcolor": GRID},
    )
    return fig
''',
    'trade_sim/cli.py': r'''"""コマンドラインのエントリーポイント。 `python -m trade_sim --help` で使い方を表示する。"""

from __future__ import annotations

import argparse
import os
import sys
from datetime import date, datetime

from . import __version__
from .instruments import INSTRUMENTS
from .portfolio import LONG, SHORT
from .prices import PriceSource
from .simulator import USER_ERRORS, Result, Simulator, now_jst, parse_date
from .table import render_table

DEFAULT_FILE = "portfolio.json"
FILE_ENV = "TRADE_SIM_FILE"


def print_result(result: Result) -> None:
    if result.table is not None:
        table = result.table
        print(render_table(table.headers, table.rows, table.aligns, table.footer))
        if result.lines or result.notes:
            print()
    for line in result.lines:
        print(line)
    for note in result.notes:
        print(f"※ {note}")


# ---- コマンド ------------------------------------------------------------


def cmd_quote(args: argparse.Namespace, sim: Simulator) -> Result:
    return sim.quote(args.symbols)


def cmd_open(args: argparse.Namespace, sim: Simulator) -> Result:
    return sim.open(
        args.symbol,
        args.side,
        args.qty,
        price=args.price,
        day=args.date,
        leverage=args.leverage,
        take_profit_yen=args.tp,
        stop_loss_yen=args.sl,
    )


def cmd_tpsl(args: argparse.Namespace, sim: Simulator) -> Result:
    return sim.set_exit(args.id, take_profit_yen=args.tp, stop_loss_yen=args.sl)


def cmd_close(args: argparse.Namespace, sim: Simulator) -> Result:
    return sim.close(args.ids, qty=args.qty, price=args.price, day=args.date)


def cmd_positions(args: argparse.Namespace, sim: Simulator) -> Result:
    return sim.positions()


def cmd_history(args: argparse.Namespace, sim: Simulator) -> Result:
    return sim.history(args.symbol)


def cmd_summary(args: argparse.Namespace, sim: Simulator) -> Result:
    return sim.summary()


def cmd_account(args: argparse.Namespace, sim: Simulator) -> Result:
    if args.capital is None and args.losscut is None:
        return sim.account()
    changed = sim.set_account(initial_capital=args.capital, losscut_level=args.losscut)
    status = sim.account()
    return Result(lines=[*changed.lines, *status.lines], notes=status.notes)


def cmd_reset(args: argparse.Namespace, sim: Simulator) -> Result:
    if sim.path.exists() and not args.yes:
        try:
            answer = input(f"{sim.path} の建玉と取引履歴をすべて削除します。よろしいですか? [y/N]: ")
        except EOFError:
            answer = ""
        if answer.strip().lower() not in ("y", "yes"):
            return Result(lines=["中止しました。"])
    return sim.reset()


# ---- 引数の解析 ----------------------------------------------------------


def _positive_int(text: str) -> int:
    try:
        value = int(text.replace(",", ""))
    except ValueError:
        raise argparse.ArgumentTypeError(f"整数で指定してください: {text}") from None
    if value <= 0:
        raise argparse.ArgumentTypeError(f"1 以上を指定してください: {text}")
    return value


def _non_negative_float(text: str) -> float:
    try:
        value = float(text.replace(",", "").rstrip("%"))
    except ValueError:
        raise argparse.ArgumentTypeError(f"数値で指定してください: {text}") from None
    if value < 0:
        raise argparse.ArgumentTypeError(f"0 以上を指定してください: {text}")
    return value


def _positive_float(text: str) -> float:
    try:
        value = float(text.replace(",", ""))
    except ValueError:
        raise argparse.ArgumentTypeError(f"数値で指定してください: {text}") from None
    if not value > 0:
        raise argparse.ArgumentTypeError(f"0 より大きい値を指定してください: {text}")
    return value


def _date(text: str) -> date:
    try:
        day = parse_date(text)
    except USER_ERRORS as e:
        raise argparse.ArgumentTypeError(str(e)) from None
    if day is None:
        raise argparse.ArgumentTypeError("日付を YYYY-MM-DD 形式で指定してください")
    return day


def _add_exit_options(parser: argparse.ArgumentParser) -> None:
    parser.add_argument(
        "--tp", type=_non_negative_float, metavar="YEN", help="この金額の利益になったら決済する (利益確定。円)"
    )
    parser.add_argument(
        "--sl", type=_non_negative_float, metavar="YEN", help="この金額の損失になったら決済する (損切り。円)"
    )


def _add_price_options(parser: argparse.ArgumentParser) -> None:
    parser.add_argument(
        "--date",
        type=_date,
        metavar="YYYY-MM-DD",
        help="その日の終値で約定させる (休場日なら直前の営業日)。省略すると最新の価格",
    )
    parser.add_argument("--price", type=_positive_float, metavar="PRICE", help="約定価格を直接指定する")


def _epilog() -> str:
    lines = ["対応銘柄 (銘柄名・コード・ティッカーのどれでも指定できます):"]
    for inst in INSTRUMENTS:
        names = " / ".join(dict.fromkeys([inst.name, inst.key.lower(), *inst.aliases]))
        lines.append(f"  {names}  [{inst.ticker}]")
    lines += [
        "",
        "例:",
        "  python -m trade_sim quote                       現在値を表示",
        "  python -m trade_sim long kioxia 100             キオクシアを 100 株ロング",
        "  python -m trade_sim short usdjpy 10000 -l 10    ドル円を 1 万通貨、レバレッジ 10 倍でショート",
        "  python -m trade_sim long usdjpy 10000 --tp 5000 --sl 3000   +5,000円で利確・-3,000円で損切りする注文つき",
        "  python -m trade_sim long 6920 100 --date 2025-04-01   過去の終値で約定",
        "  python -m trade_sim positions                   保有中の建玉と評価損益",
        "  python -m trade_sim close 1                     建玉 #1 を決済",
        "  python -m trade_sim tpsl 1 --tp 20000 --sl 0    建玉 #1 の利確を +20,000円 に変え、損切りを取り消す",
        "  python -m trade_sim summary                     通算損益を表示",
        "  python -m trade_sim account                     資金・証拠金・余力・証拠金維持率を表示",
    ]
    return "\n".join(lines)


def build_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(
        prog="python -m trade_sim",
        description="yfinance の価格で、株・為替のロング/ショート取引をシミュレーションします。",
        epilog=_epilog(),
        formatter_class=argparse.RawDescriptionHelpFormatter,
    )
    parser.add_argument(
        "--file",
        default=os.environ.get(FILE_ENV, DEFAULT_FILE),
        metavar="PATH",
        help=f"取引データの保存先 (既定: {DEFAULT_FILE}。環境変数 {FILE_ENV} でも指定可)",
    )
    parser.add_argument("--version", action="version", version=f"%(prog)s {__version__}")
    sub = parser.add_subparsers(dest="command", metavar="COMMAND")

    p = sub.add_parser("quote", help="現在値を表示する")
    p.add_argument("symbols", nargs="*", metavar="SYMBOL", help="銘柄 (省略すると全銘柄)")
    p.set_defaults(func=cmd_quote)

    for side, text in ((LONG, "新規にロング (買い) で建てる"), (SHORT, "新規にショート (売り) で建てる")):
        p = sub.add_parser(side, help=text, description=text)
        p.add_argument("symbol", metavar="SYMBOL", help="銘柄 (例: kioxia、6920、ドル円、eurjpy)")
        p.add_argument("qty", type=_positive_int, metavar="QTY", help="数量。株数または通貨数 (例: 100、10000)")
        p.add_argument(
            "-l",
            "--leverage",
            type=_positive_float,
            default=1.0,
            metavar="X",
            help="レバレッジ (株は 3.3 倍、為替は 500 倍まで。既定: 1 倍)",
        )
        _add_exit_options(p)
        _add_price_options(p)
        p.set_defaults(func=cmd_open, side=side)

    p = sub.add_parser("close", help="建玉を決済する", description="建玉を決済する")
    p.add_argument("ids", nargs="+", metavar="ID", help="決済する建玉の ID (複数可)。all ですべての建玉")
    p.add_argument("--qty", type=_positive_int, metavar="QTY", help="一部だけ決済する数量")
    _add_price_options(p)
    p.set_defaults(func=cmd_close)

    p = sub.add_parser("positions", help="保有中の建玉と評価損益を表示する")
    p.set_defaults(func=cmd_positions)

    p = sub.add_parser("history", help="決済済みの取引履歴を表示する")
    p.add_argument("symbol", nargs="?", metavar="SYMBOL", help="銘柄 (指定するとその銘柄だけ表示)")
    p.set_defaults(func=cmd_history)

    p = sub.add_parser("summary", aliases=["pnl"], help="銘柄別と通算の損益を表示する")
    p.set_defaults(func=cmd_summary)

    p = sub.add_parser(
        "tpsl",
        help="保有中の建玉に決済注文 (利益確定・損切り) を出す・変える",
        description=(
            "保有中の建玉に、指定した金額の利益・損失になったら決済する注文を出します。"
            "指定しなかった方は今の注文のまま、0 を指定するとその注文を取り消します。"
        ),
    )
    p.add_argument("id", metavar="ID", help="建玉の ID")
    _add_exit_options(p)
    p.set_defaults(func=cmd_tpsl)

    p = sub.add_parser(
        "account",
        help="資金・証拠金・余力・証拠金維持率を表示する (初期資金やロスカット水準の変更も)",
        description="口座の状態を表示します。オプションを付けると設定を変えます。",
    )
    p.add_argument("--capital", type=_positive_float, metavar="YEN", help="初期資金を変える (例: 10000000)")
    p.add_argument(
        "--losscut",
        type=_non_negative_float,
        metavar="PERCENT",
        help="ロスカット水準 (証拠金維持率 %%) を変える。0 ならロスカットしない (既定: 50)",
    )
    p.set_defaults(func=cmd_account)

    p = sub.add_parser("reset", help="建玉と取引履歴をすべて削除する")
    p.add_argument("-y", "--yes", action="store_true", help="確認せずに削除する")
    p.set_defaults(func=cmd_reset)

    return parser


def main(
    argv: list[str] | None = None,
    source: PriceSource | None = None,
    now: datetime | None = None,
) -> int:
    parser = build_parser()
    args = parser.parse_args(argv)
    if args.command is None:
        parser.print_help()
        return 0
    clock = (lambda: now) if now is not None else now_jst
    sim = Simulator(args.file, source, clock)
    try:
        result = args.func(args, sim)
    except USER_ERRORS as e:
        print(f"エラー: {e}", file=sys.stderr)
        return 1
    print_result(result)
    return 0 if result.ok else 1
''',
    'trade_sim/instruments.py': r'''"""シミュレーション対象の銘柄の定義。

銘柄を増やすときは INSTRUMENTS に Instrument を追加する。
損益はすべて円で計算するので、円建ての銘柄 (東証の株、XXX/JPY の為替) を追加すること。
"""

from __future__ import annotations

import unicodedata
from dataclasses import dataclass


# レバレッジの上限。株は日本の信用取引 (保証金率 30% で約 3.3 倍)、
# 為替は海外の FX 業者並みの 500 倍まで (日本の FX の個人向けの上限は 25 倍)
MAX_LEVERAGE = {"stock": 3.3, "fx": 500.0}
# 画面で選べるレバレッジ
LEVERAGE_CHOICES = {
    "stock": (1.0, 1.5, 2.0, 2.5, 3.0, 3.3),
    "fx": (1.0, 2.0, 3.0, 5.0, 10.0, 15.0, 20.0, 25.0, 50.0, 100.0, 200.0, 300.0, 400.0, 500.0),
}


def format_leverage(leverage: float) -> str:
    return f"{leverage:g}倍"


@dataclass(frozen=True)
class Instrument:
    key: str  # 内部 ID (保存データにはこの値が入る)
    name: str  # 表示名
    ticker: str  # yfinance (Yahoo Finance) のティッカー
    kind: str  # "stock" (株式) / "fx" (為替)
    unit: str  # 数量の単位
    price_decimals: int  # 価格の小数点以下の桁数
    aliases: tuple[str, ...] = ()  # コマンドで指定するときの別名

    def format_price(self, price: float) -> str:
        return f"{price:,.{self.price_decimals}f}"

    def format_qty(self, qty: int) -> str:
        return f"{qty:,}{self.unit}"

    @property
    def max_leverage(self) -> float:
        return MAX_LEVERAGE[self.kind]


INSTRUMENTS: tuple[Instrument, ...] = (
    Instrument("KIOXIA", "キオクシア", "285A.T", "stock", "株", 1, ("285A",)),
    Instrument("LASERTEC", "レーザーテック", "6920.T", "stock", "株", 1, ("6920",)),
    Instrument("USDJPY", "ドル円", "USDJPY=X", "fx", "通貨", 3, ("USD/JPY", "ドル/円")),
    Instrument("EURJPY", "ユーロ円", "EURJPY=X", "fx", "通貨", 3, ("EUR/JPY", "ユーロ/円")),
)


class UnknownInstrumentError(ValueError):
    pass


def _normalize(text: str) -> str:
    # 全角英数字や半角カナでも指定できるように NFKC で正規化する
    return unicodedata.normalize("NFKC", text).strip().lower()


_BY_KEY = {inst.key: inst for inst in INSTRUMENTS}
_LOOKUP = {
    _normalize(name): inst
    for inst in INSTRUMENTS
    for name in (inst.key, inst.name, inst.ticker, *inst.aliases)
}


def get_instrument(key: str) -> Instrument:
    """保存データ内の ID から銘柄を引く。"""
    try:
        return _BY_KEY[key]
    except KeyError:
        raise UnknownInstrumentError(f"未対応の銘柄 ID です: {key}") from None


def find_instrument(text: str) -> Instrument:
    """コマンドで入力された銘柄名・コード・別名から銘柄を探す。"""
    inst = _LOOKUP.get(_normalize(text))
    if inst is None:
        choices = "、".join(f"{i.name}({i.key.lower()})" for i in INSTRUMENTS)
        raise UnknownInstrumentError(f"銘柄「{text}」は対応していません。対応銘柄: {choices}")
    return inst
''',
    'trade_sim/notebook.py': r'''"""Google Colab / Jupyter Notebook 用の取引画面 (ipywidgets)。

ノートブックのセルで次のように実行すると、注文・決済・損益確認ができる画面が出る。

    from trade_sim.notebook import show
    app = show("portfolio.json")
"""

from __future__ import annotations

import html
from collections.abc import Callable

import ipywidgets as widgets
from IPython.display import display

from .instruments import INSTRUMENTS, LEVERAGE_CHOICES, Instrument, format_leverage, get_instrument
from .portfolio import LONG, SHORT, SIDE_LABELS
from .prices import PriceSource
from .simulator import USER_ERRORS, Result, Simulator, Table, parse_date

# 銘柄を選んだときの数量の初期値
DEFAULT_QTY = {"stock": 100, "fx": 10000}

# Colab のライト/ダーク両方で読めるよう、文字色は指定せず半透明の線だけ使う
_LINE = "rgba(127, 127, 127, 0.35)"
_ERROR = "#d9534f"


# ---- HTML の組み立て -----------------------------------------------------


def table_html(table: Table) -> str:
    def cells(row: list[str], tag: str, extra: str = "") -> str:
        out = []
        for text, align in zip(row, table.aligns):
            style = (
                f"padding: 4px 10px; border-bottom: 1px solid {_LINE}; white-space: nowrap;"
                f" text-align: {'right' if align == '>' else 'left'};{extra}"
            )
            out.append(f'<{tag} style="{style}">{html.escape(text)}</{tag}>')
        return "".join(out)

    parts = [
        '<div style="overflow-x: auto; max-width: 100%;">',
        # color / font-size: inherit は、互換モードの表示でも表がまわりの文字色・大きさを引き継ぐようにするため
        '<table style="border-collapse: collapse; color: inherit; font-size: inherit;'
        ' font-variant-numeric: tabular-nums; margin: 2px 0 6px;">',
        f"<thead><tr>{cells(table.headers, 'th', ' font-weight: 600;')}</tr></thead>",
        "<tbody>",
        *(f"<tr>{cells(row, 'td')}</tr>" for row in table.rows),
        "</tbody>",
    ]
    if table.footer:
        footer_style = f" font-weight: 600; border-top: 2px solid {_LINE};"
        parts += ["<tfoot>", *(f"<tr>{cells(row, 'td', footer_style)}</tr>" for row in table.footer), "</tfoot>"]
    parts += ["</table>", "</div>"]
    return "".join(parts)


def result_html(result: Result, emphasize: bool = False) -> str:
    """操作結果を HTML にする。emphasize=True なら最初の行 (通算損益など) を大きく表示する。"""
    parts = []
    if result.table is not None:
        parts.append(table_html(result.table))
    for i, line in enumerate(result.lines):
        style = "font-size: 1.2em; font-weight: 700; margin: 4px 0;" if emphasize and i == 0 else "margin: 2px 0;"
        parts.append(f'<div style="{style}">{html.escape(line)}</div>')
    for note in result.notes:
        parts.append(f'<div style="margin: 2px 0; opacity: 0.75; font-size: 0.92em;">※ {html.escape(note)}</div>')
    return "".join(parts)


def error_html(message: str) -> str:
    return f'<div style="color: {_ERROR}; font-weight: 600; margin: 2px 0;">エラー: {html.escape(message)}</div>'


def _info_html(message: str) -> str:
    return f'<div style="margin: 2px 0; opacity: 0.75;">{html.escape(message)}</div>'


def _heading(text: str) -> widgets.HTML:
    style = f"font-size: 1.1em; font-weight: 700; margin: 16px 0 2px; padding-bottom: 2px; border-bottom: 1px solid {_LINE};"
    return widgets.HTML(f'<div style="{style}">{html.escape(text)}</div>')


def _leverage_options(inst: Instrument) -> list[tuple[str, float]]:
    return [(format_leverage(x), x) for x in LEVERAGE_CHOICES[inst.kind]]


def _row(*children: widgets.Widget) -> widgets.HBox:
    return widgets.HBox(list(children), layout=widgets.Layout(flex_flow="row wrap", align_items="center"))


# ---- 画面 ----------------------------------------------------------------


class TradeApp:
    def __init__(self, simulator: Simulator) -> None:
        self.sim = simulator
        label = {"description_width": "5.5em"}
        first = INSTRUMENTS[0]

        # 現在値
        self.status = widgets.HTML()
        self.quote_view = widgets.HTML()
        self.refresh_button = widgets.Button(description="価格を更新", icon="refresh")

        # 新規注文
        self.symbol = widgets.Dropdown(
            options=[(inst.name, inst.key) for inst in INSTRUMENTS], value=first.key, description="銘柄", style=label
        )
        self.side = widgets.ToggleButtons(
            options=[("ロング（買い）", LONG), ("ショート（売り）", SHORT)], value=LONG, description="売買", style=label
        )
        self.qty = widgets.IntText(
            value=DEFAULT_QTY[first.kind], description="数量", style=label, layout=widgets.Layout(width="15em")
        )
        self.unit = widgets.HTML(first.unit)
        self.leverage = widgets.Dropdown(
            options=_leverage_options(first), value=1.0, description="レバレッジ", style=label,
            layout=widgets.Layout(width="15em"),
        )
        self.leverage_hint = widgets.HTML(f"{format_leverage(first.max_leverage)}まで")
        self.take_profit = widgets.IntText(
            value=0, description="利益確定", style=label, layout=widgets.Layout(width="15em")
        )
        self.stop_loss = widgets.IntText(value=0, description="損切り", style=label, layout=widgets.Layout(width="15em"))
        self.order_date = widgets.Text(
            placeholder="空欄なら最新の価格（例: 2025-04-01）",
            description="約定日",
            style=label,
            layout=widgets.Layout(width="26em"),
        )
        self.order_button = widgets.Button(description="発注する", button_style="primary")
        self.order_message = widgets.HTML()

        # 決済
        self.positions_view = widgets.HTML()
        self.close_target = widgets.Dropdown(description="建玉", style=label, layout=widgets.Layout(width="30em"))
        self.close_qty = widgets.IntText(value=0, description="数量", style=label, layout=widgets.Layout(width="15em"))
        self.close_date = widgets.Text(
            placeholder="空欄なら最新の価格", description="決済日", style=label, layout=widgets.Layout(width="26em")
        )
        self.close_button = widgets.Button(description="決済する", button_style="warning")
        self.close_all_button = widgets.Button(description="すべて決済")
        self.close_message = widgets.HTML()

        # 損益・履歴
        self.summary_view = widgets.HTML()
        self.history_view = widgets.HTML()

        self._busy = False
        self._has_positions = False
        self._confirming_close_all = False

        self.symbol.observe(self._on_symbol_change, names="value")
        self.refresh_button.on_click(self._on_refresh)
        self.order_button.on_click(self._on_order)
        self.close_button.on_click(self._on_close)
        self.close_all_button.on_click(self._on_close_all)

        self.widget = widgets.VBox(
            [
                widgets.HTML('<div style="font-size: 1.4em; font-weight: 700;">株・為替 取引シミュレーター</div>'),
                self.status,
                _heading("損益"),
                self.summary_view,
                _heading("現在値"),
                self.quote_view,
                self.refresh_button,
                _heading("新規注文"),
                self.symbol,
                self.side,
                _row(self.qty, self.unit),
                _row(self.leverage, self.leverage_hint),
                _row(self.take_profit, widgets.HTML("円")),
                _row(self.stop_loss, widgets.HTML("円（0 なら注文しない。この金額の利益・損失になったら決済します）")),
                self.order_date,
                self.order_button,
                self.order_message,
                _heading("保有中の建玉"),
                self.positions_view,
                self.close_target,
                _row(self.close_qty, widgets.HTML("0 ならすべて決済")),
                self.close_date,
                _row(self.close_button, self.close_all_button),
                self.close_message,
                _heading("取引履歴"),
                self.history_view,
            ]
        )
        self.refresh()

    def refresh(self) -> None:
        """表示をすべて最新の状態にする。"""
        views: list[tuple[widgets.HTML, Callable[[], str]]] = [
            (self.summary_view, lambda: result_html(self.sim.summary(), emphasize=True)),
            (self.quote_view, lambda: result_html(self.sim.quote())),
            (self.positions_view, lambda: result_html(self.sim.positions())),
            (self.history_view, lambda: result_html(self.sim.history())),
        ]
        for view, render in views:
            try:
                view.value = render()
            except USER_ERRORS as e:
                view.value = error_html(str(e))
        self._update_close_targets()
        self.status.value = _info_html(
            f"最終更新: {self.sim.clock():%Y-%m-%d %H:%M}（日本時間）　データ: {self.sim.path}"
        )

    # ---- イベント --------------------------------------------------------

    def _on_symbol_change(self, change: dict) -> None:
        inst = get_instrument(change["new"])
        self.qty.value = DEFAULT_QTY[inst.kind]
        self.unit.value = inst.unit
        self.leverage.options = _leverage_options(inst)
        self.leverage.value = 1.0
        self.leverage_hint.value = f"{format_leverage(inst.max_leverage)}まで"

    def _on_refresh(self, _button: widgets.Button) -> None:
        self._cancel_close_all()
        self._set_busy(True)
        try:
            self.sim.refresh_prices()
            self.refresh()
        finally:
            self._set_busy(False)

    def _on_order(self, _button: widgets.Button) -> None:
        self._cancel_close_all()
        symbol, side, qty, day_text = self.symbol.value, self.side.value, self.qty.value, self.order_date.value
        leverage = float(self.leverage.value)
        tp, sl = self.take_profit.value or None, self.stop_loss.value or None
        self._run(
            self.order_message,
            lambda: self.sim.open(
                symbol, side, qty, day=parse_date(day_text), leverage=leverage, take_profit_yen=tp, stop_loss_yen=sl
            ),
        )

    def _on_close(self, _button: widgets.Button) -> None:
        self._cancel_close_all()
        target = self.close_target.value
        if target is None:
            self.close_message.value = error_html("決済する建玉を選んでください。")
            return
        qty = self.close_qty.value or None  # 0 ならすべて決済
        day_text = self.close_date.value
        self._run(self.close_message, lambda: self.sim.close([target], qty=qty, day=parse_date(day_text)))

    def _on_close_all(self, _button: widgets.Button) -> None:
        if not self._confirming_close_all:
            self._confirming_close_all = True
            self.close_all_button.description = "もう一度押すと全決済"
            self.close_all_button.button_style = "danger"
            self.close_message.value = _info_html(
                "保有中の建玉をすべて決済します。よければ「もう一度押すと全決済」を押してください。"
            )
            return
        self._cancel_close_all()
        day_text = self.close_date.value
        self._run(self.close_message, lambda: self.sim.close(["all"], day=parse_date(day_text)))

    # ---- 内部 ------------------------------------------------------------

    def _run(self, message: widgets.HTML, action: Callable[[], Result]) -> None:
        """ボタンの操作を実行して、結果かエラーを message に表示する。"""
        self._set_busy(True)
        message.value = _info_html("処理中…")
        try:
            result = action()
        except USER_ERRORS as e:
            message.value = error_html(str(e))
        except Exception as e:  # ボタンから呼ばれるので、想定外のエラーも画面に出す
            message.value = error_html(f"予期しないエラーが発生しました ({type(e).__name__}: {e})")
        else:
            message.value = result_html(result)
            self.refresh()
        finally:
            self._set_busy(False)

    def _cancel_close_all(self) -> None:
        if self._confirming_close_all:
            self._confirming_close_all = False
            self.close_all_button.description = "すべて決済"
            self.close_all_button.button_style = ""

    def _update_close_targets(self) -> None:
        try:
            positions = self.sim.load().positions
        except USER_ERRORS:
            positions = []
        options = []
        for p in positions:
            inst = get_instrument(p.symbol)
            label = (
                f"#{p.id} {inst.name} {SIDE_LABELS[p.side]} {inst.format_qty(p.qty)}"
                f" @ {inst.format_price(p.entry_price)}"
            )
            options.append((label, p.id))
        current = self.close_target.value
        self.close_target.options = options
        values = [value for _, value in options]
        if values:
            # 選んでいた建玉が残っていればそのまま、なければ先頭を選ぶ (空の状態から増えたときは未選択になるため)
            self.close_target.value = current if current in values else values[0]
        self._has_positions = bool(options)
        self._update_buttons()

    def _set_busy(self, busy: bool) -> None:
        self._busy = busy
        self._update_buttons()

    def _update_buttons(self) -> None:
        self.refresh_button.disabled = self._busy
        self.order_button.disabled = self._busy
        can_close = self._has_positions and not self._busy
        self.close_target.disabled = not self._has_positions
        self.close_button.disabled = not can_close
        self.close_all_button.disabled = not can_close


def _enable_colab_widgets() -> None:
    """Colab で ipywidgets 8 以降を使う場合は、Colab のカスタムウィジェット表示を有効にする。"""
    try:
        from google.colab import output  # type: ignore[import-not-found]
    except ImportError:
        return
    if int(widgets.__version__.split(".")[0]) >= 8:
        output.enable_custom_widget_manager()


def show(data_file: str = "portfolio.json", source: PriceSource | None = None) -> TradeApp:
    """取引画面を表示する。data_file は取引データの保存先。"""
    _enable_colab_widgets()
    app = TradeApp(Simulator(data_file, source))
    display(app.widget)
    return app
''',
    'trade_sim/portfolio.py': r'''"""建玉と決済履歴の管理、損益と証拠金の計算。

損益はすべて円で計算する (対象銘柄はすべて円建てのため)。
  ロング:   (決済値 - 建値) × 数量
  ショート: (建値 - 決済値) × 数量

レバレッジは、建てるのに必要な証拠金を決める (損益そのものは変わらない)。
  必要証拠金     = 建値 × 数量 ÷ レバレッジ
  資金           = 初期資金 + 実現損益
  有効証拠金     = 資金 + 評価損益
  余力           = 有効証拠金 - 必要証拠金
  証拠金維持率   = 有効証拠金 ÷ 必要証拠金 × 100 (%)

建玉には決済注文 (利益確定の価格・損切りの価格) を付けられる。値動きがその価格に届いたら決済する (find_exit)。
"""

from __future__ import annotations

import json
import os
from collections.abc import Iterable
from dataclasses import asdict, dataclass
from datetime import datetime, timedelta
from pathlib import Path

from .instruments import INSTRUMENTS, format_leverage, get_instrument

LONG = "long"
SHORT = "short"
SIDE_LABELS = {LONG: "ロング", SHORT: "ショート"}
DATA_VERSION = 2

DEFAULT_CAPITAL = 10_000_000  # 初期資金 (円)
DEFAULT_LOSSCUT_LEVEL = 50.0  # 証拠金維持率 (%) がこれを下回ったらロスカット。0 ならロスカットしない
# Trade.reason: 決済の理由 (手動の決済は "")
LOSSCUT = "losscut"
TAKE_PROFIT = "take_profit"
STOP_LOSS = "stop_loss"
REASON_LABELS = {LOSSCUT: "ロスカット", TAKE_PROFIT: "利確", STOP_LOSS: "損切り"}


class PortfolioError(Exception):
    pass


def parse_trade_time(text: str) -> datetime:
    """記録された日時を datetime にする。日付だけのもの (終値で約定) はその日の取引終了後として扱う。"""
    if len(text) <= 10:
        return datetime.fromisoformat(text) + timedelta(hours=23, minutes=59)
    return datetime.fromisoformat(text)


def calc_pnl(side: str, entry_price: float, exit_price: float, qty: int) -> float:
    diff = exit_price - entry_price if side == LONG else entry_price - exit_price
    # 浮動小数点の誤差 (例: 22220.000000000004) を落とすため銭単位で丸める
    return round(diff * qty, 2)


@dataclass
class Position:
    """保有中の建玉。"""

    id: int
    symbol: str
    side: str
    qty: int
    entry_price: float
    opened_at: str  # "YYYY-MM-DD HH:MM" (現在値で約定) または "YYYY-MM-DD" (終値で約定)
    leverage: float = 1.0
    take_profit: float | None = None  # 利益確定の価格
    stop_loss: float | None = None  # 損切りの価格
    exit_set_at: str | None = None  # 決済注文を出した (変えた) 日時。これより後の値動きで判定する

    def unrealized_pnl(self, price: float) -> float:
        return calc_pnl(self.side, self.entry_price, price, self.qty)

    def exit_pnl(self, price: float | None) -> float | None:
        """決済注文の価格で決済した場合の損益。"""
        return None if price is None else self.unrealized_pnl(price)

    @property
    def required_margin(self) -> float:
        return required_margin(self.entry_price, self.qty, self.leverage)


@dataclass
class Trade:
    """決済済みの取引。一部決済した場合は決済ごとに 1 件。"""

    position_id: int
    symbol: str
    side: str
    qty: int
    entry_price: float
    exit_price: float
    opened_at: str
    closed_at: str
    leverage: float = 1.0
    reason: str = ""  # 決済の理由 (REASON_LABELS)。手動の決済は ""

    @property
    def pnl(self) -> float:
        return calc_pnl(self.side, self.entry_price, self.exit_price, self.qty)


@dataclass
class SummaryRow:
    """銘柄ごとの損益集計。"""

    symbol: str
    trades: int  # 決済回数
    wins: int  # 利益が出た決済の回数
    realized: float  # 実現損益
    open_positions: int  # 保有中の建玉数
    unrealized: float | None  # 評価損益 (現在値が取得できなかった場合は None)

    @property
    def win_rate(self) -> float | None:
        return self.wins / self.trades if self.trades else None

    @property
    def total(self) -> float | None:
        return None if self.unrealized is None else round(self.realized + self.unrealized, 2)


@dataclass
class AccountStatus:
    """口座の状態。"""

    initial_capital: float
    realized: float
    unrealized: float | None  # 現在値を取得できない建玉があれば None
    used_margin: float  # 必要証拠金の合計
    losscut_level: float

    @property
    def cash(self) -> float:
        """資金 (初期資金 + 実現損益)。"""
        return round(self.initial_capital + self.realized, 2)

    @property
    def equity(self) -> float | None:
        """有効証拠金 (資金 + 評価損益)。"""
        return None if self.unrealized is None else round(self.cash + self.unrealized, 2)

    @property
    def free_margin(self) -> float | None:
        """余力 (新しく建てるのに使える証拠金)。"""
        return None if self.equity is None else round(self.equity - self.used_margin, 2)

    @property
    def margin_level(self) -> float | None:
        """証拠金維持率 (%)。建玉が無ければ None。"""
        if self.equity is None or self.used_margin <= 0:
            return None
        return self.equity / self.used_margin * 100

    @property
    def losscut_triggered(self) -> bool:
        level = self.margin_level
        return self.losscut_level > 0 and level is not None and level < self.losscut_level


def find_exit(
    position: Position,
    candles: Iterable[tuple[datetime, float, float, float]],
    latest: float | None = None,
    now: datetime | None = None,
) -> tuple[datetime, float, str] | None:
    """決済注文が約定する最初の時点を探す。戻り値は (時刻, 約定価格, TAKE_PROFIT か STOP_LOSS)。

    candles は決済注文を出した後のローソク足 (時刻, 始値, 高値, 安値) を時刻順に並べたもの。
    - 始値の時点で注文の価格を飛び越えていたら (窓開け)、始値で約定する
    - 同じ足の中で利益確定と損切りの両方の価格に届いたら、不利な方 (損切り) が先とみなす
    - ローソク足が無い時間の分は latest (現在値) でも確かめる。届いていれば注文の価格で約定する
    """
    tp, sl = position.take_profit, position.stop_loss
    if tp is None and sl is None:
        return None
    long = position.side == LONG

    def beyond_sl(price: float) -> bool:
        return sl is not None and (price <= sl if long else price >= sl)

    def beyond_tp(price: float) -> bool:
        return tp is not None and (price >= tp if long else price <= tp)

    for when, open_, high, low in candles:
        if beyond_sl(open_):
            return when, open_, STOP_LOSS
        if beyond_tp(open_):
            return when, open_, TAKE_PROFIT
        if beyond_sl(low if long else high):
            return when, sl, STOP_LOSS
        if beyond_tp(high if long else low):
            return when, tp, TAKE_PROFIT
    if latest is not None and now is not None:
        if beyond_sl(latest):
            return now, sl, STOP_LOSS
        if beyond_tp(latest):
            return now, tp, TAKE_PROFIT
    return None


def exit_prices(
    position: Position, take_profit_yen: float | None, stop_loss_yen: float | None
) -> tuple[float | None, float | None]:
    """利益確定・損切りの金額 (円) を、その金額になる価格に直す。None か 0 なら注文なし。"""
    sign = 1 if position.side == LONG else -1
    tp = sl = None
    if take_profit_yen:
        tp = position.entry_price + sign * take_profit_yen / position.qty
    if stop_loss_yen:
        sl = position.entry_price - sign * stop_loss_yen / position.qty
    for name, price in (("利益確定", tp), ("損切り", sl)):
        if price is not None and price <= 0:
            raise PortfolioError(f"{name}の金額が大きすぎます (価格が 0 以下になります)。")
    return tp, sl


def required_margin(price: float, qty: int, leverage: float) -> float:
    return round(price * qty / leverage, 2)


def _check_leverage(leverage: object, symbol: str) -> None:
    inst = get_instrument(symbol)
    if isinstance(leverage, bool) or not isinstance(leverage, (int, float)) or not 1 <= leverage <= inst.max_leverage:
        raise PortfolioError(
            f"{inst.name}のレバレッジは 1 倍から {format_leverage(inst.max_leverage)}までで指定してください: {leverage}"
        )


def _check_exit_amounts(*amounts: float | None) -> None:
    for amount in amounts:
        if amount is not None and amount < 0:
            raise PortfolioError(f"利益確定・損切りの金額は 0 以上で指定してください: {amount}")


def _check_qty(qty: object) -> None:
    if isinstance(qty, bool) or not isinstance(qty, int) or qty <= 0:
        raise PortfolioError(f"数量は 1 以上の整数で指定してください: {qty}")


def _check_price(price: float) -> None:
    if not price > 0:
        raise PortfolioError(f"価格が不正です: {price}")


class Portfolio:
    def __init__(
        self,
        positions: list[Position] | None = None,
        trades: list[Trade] | None = None,
        next_id: int = 1,
        initial_capital: float = DEFAULT_CAPITAL,
        losscut_level: float = DEFAULT_LOSSCUT_LEVEL,
    ) -> None:
        self.positions = positions if positions is not None else []
        self.trades = trades if trades is not None else []
        self.next_id = next_id
        self.initial_capital = initial_capital
        self.losscut_level = losscut_level

    # ---- 保存・読み込み ------------------------------------------------

    @classmethod
    def load(cls, path: str | os.PathLike) -> Portfolio:
        path = Path(path)
        if not path.exists():
            return cls()
        try:
            data = json.loads(path.read_text(encoding="utf-8"))
            positions = [Position(**p) for p in data.get("positions", [])]
            trades = [Trade(**t) for t in data.get("trades", [])]
            next_id = int(data.get("next_id", 1))
            if "account" in data:
                initial_capital = float(data["account"]["initial_capital"])
                losscut_level = float(data["account"]["losscut_level"])
            else:
                # レバレッジ導入前 (バージョン 1) のデータ。資金を考えずに建てた建玉があり得るので、
                # 読み込んだとたんにロスカットされないよう、ロスカットはオフで始める
                initial_capital, losscut_level = DEFAULT_CAPITAL, 0.0
        except (OSError, ValueError, TypeError, AttributeError, KeyError) as e:
            raise PortfolioError(f"データファイル {path} を読み込めません: {e}") from e
        return cls(positions, trades, next_id, initial_capital, losscut_level)

    def save(self, path: str | os.PathLike) -> None:
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        data = {
            "version": DATA_VERSION,
            "next_id": self.next_id,
            "account": {"initial_capital": self.initial_capital, "losscut_level": self.losscut_level},
            "positions": [asdict(p) for p in self.positions],
            "trades": [asdict(t) for t in self.trades],
        }
        # 書き込み途中で中断してもデータが壊れないよう、一時ファイルに書いてから置き換える
        tmp = path.with_name(path.name + ".tmp")
        tmp.write_text(json.dumps(data, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
        os.replace(tmp, path)

    # ---- 取引 ----------------------------------------------------------

    def open(
        self,
        symbol: str,
        side: str,
        qty: int,
        price: float,
        at: str,
        leverage: float = 1.0,
        take_profit_yen: float | None = None,
        stop_loss_yen: float | None = None,
    ) -> Position:
        """新規に建玉を作る (証拠金の余力は呼び出し側で確認する)。

        take_profit_yen / stop_loss_yen を渡すと、その金額の利益・損失で決済する注文も出す。
        """
        get_instrument(symbol)
        if side not in SIDE_LABELS:
            raise PortfolioError(f"売買区分が不正です: {side}")
        _check_qty(qty)
        _check_price(price)
        _check_leverage(leverage, symbol)
        position = Position(self.next_id, symbol, side, qty, price, at, float(leverage))
        _check_exit_amounts(take_profit_yen, stop_loss_yen)
        tp, sl = exit_prices(position, take_profit_yen, stop_loss_yen)
        if tp is not None or sl is not None:
            position.take_profit, position.stop_loss, position.exit_set_at = tp, sl, at
        self.next_id += 1
        self.positions.append(position)
        return position

    def set_exit(self, position_id: int, take_profit: float | None, stop_loss: float | None, at: str) -> Position:
        """決済注文の価格を設定する (None なら取り消し)。at より後の値動きで判定する。"""
        position = self.find(position_id)
        for price in (take_profit, stop_loss):
            if price is not None:
                _check_price(price)
        position.take_profit, position.stop_loss = take_profit, stop_loss
        position.exit_set_at = at if take_profit is not None or stop_loss is not None else None
        return position

    def find(self, position_id: int) -> Position:
        for position in self.positions:
            if position.id == position_id:
                return position
        raise PortfolioError(f"建玉 #{position_id} は見つかりません (保有中の建玉は positions で確認できます)。")

    def close(self, position_id: int, price: float, at: str, qty: int | None = None, reason: str = "") -> Trade:
        """建玉を決済する。qty を指定すると一部だけ決済する。"""
        position = self.find(position_id)
        qty = position.qty if qty is None else qty
        _check_qty(qty)
        _check_price(price)
        if qty > position.qty:
            raise PortfolioError(f"決済数量 {qty:,} が建玉 #{position_id} の数量 {position.qty:,} を超えています。")
        if at[:10] < position.opened_at[:10]:
            raise PortfolioError(
                f"決済日 {at[:10]} が建玉 #{position_id} の約定日 {position.opened_at[:10]} より前です。"
            )

        trade = Trade(
            position.id,
            position.symbol,
            position.side,
            qty,
            position.entry_price,
            price,
            position.opened_at,
            at,
            position.leverage,
            reason,
        )
        self.trades.append(trade)
        position.qty -= qty
        if position.qty == 0:
            self.positions.remove(position)
        return trade

    # ---- 集計 ----------------------------------------------------------

    @property
    def realized_pnl(self) -> float:
        return round(sum(t.pnl for t in self.trades), 2)

    def set_account(self, initial_capital: float | None = None, losscut_level: float | None = None) -> None:
        if initial_capital is not None:
            if not initial_capital > 0:
                raise PortfolioError(f"初期資金は 0 より大きい金額にしてください: {initial_capital}")
            self.initial_capital = float(initial_capital)
        if losscut_level is not None:
            if not 0 <= losscut_level <= 100:
                raise PortfolioError(f"ロスカット水準は 0〜100% で指定してください (0 ならロスカットしない): {losscut_level}")
            self.losscut_level = float(losscut_level)

    def status(self, prices: dict[str, float | None]) -> AccountStatus:
        """口座の状態。prices は銘柄 ID -> 現在値 (保有中の銘柄の分)。"""
        unrealized: float | None = 0.0
        for position in self.positions:
            price = prices.get(position.symbol)
            if price is None:
                unrealized = None
                break
            unrealized += position.unrealized_pnl(price)
        return AccountStatus(
            initial_capital=self.initial_capital,
            realized=self.realized_pnl,
            unrealized=None if unrealized is None else round(unrealized, 2),
            used_margin=round(sum(p.required_margin for p in self.positions), 2),
            losscut_level=self.losscut_level,
        )

    def summary(self, prices: dict[str, float | None]) -> list[SummaryRow]:
        """銘柄ごとの損益。prices は銘柄 ID -> 現在値 (保有中の銘柄の分だけあればよい)。"""
        rows = []
        for inst in INSTRUMENTS:
            trades = [t for t in self.trades if t.symbol == inst.key]
            positions = [p for p in self.positions if p.symbol == inst.key]
            price = prices.get(inst.key)
            if not positions:
                unrealized: float | None = 0.0
            elif price is None:
                unrealized = None
            else:
                unrealized = round(sum(p.unrealized_pnl(price) for p in positions), 2)
            rows.append(
                SummaryRow(
                    symbol=inst.key,
                    trades=len(trades),
                    wins=sum(1 for t in trades if t.pnl > 0),
                    realized=round(sum(t.pnl for t in trades), 2),
                    open_positions=len(positions),
                    unrealized=unrealized,
                )
            )
        return rows
''',
    'trade_sim/prices.py': r'''"""yfinance を使った価格の取得。"""

from __future__ import annotations

import logging
import time
from dataclasses import dataclass
from datetime import date, datetime, timedelta, timezone
from typing import TYPE_CHECKING, Protocol

from .instruments import Instrument

if TYPE_CHECKING:
    import pandas as pd

# 指定日が休場日だったときに、直前の営業日を探してさかのぼる日数
LOOKBACK_DAYS = 14

# 最新の価格を使い回す秒数 (Colab のように開きっぱなしで使う場合も、これより古い価格は取得し直す)
LATEST_CACHE_SECONDS = 60


class PriceError(Exception):
    """価格を取得できなかった。"""


# yfinance の period を短い順に並べたもの
PERIOD_ORDER = ("1d", "5d", "1mo", "3mo", "6mo", "1y", "2y", "5y", "10y", "max")


@dataclass(frozen=True)
class Timeframe:
    """チャートの足の種類。"""

    key: str
    label: str
    interval: str  # yfinance の interval
    periods: tuple[tuple[str, str], ...]  # (表示名, yfinance の period)
    default_period: str
    ma_windows: tuple[int, ...]  # 移動平均線の本数
    intraday: bool
    max_fetch: str  # 取得できる最長の period (Yahoo の制限: 5分足・15分足は 60 日、1時間足は 730 日まで)

    def fetch_period(self, period: str) -> str:
        """移動平均線をチャートの左端から描けるよう、表示する期間より 1 段階長く取得する。"""
        longer = PERIOD_ORDER[min(PERIOD_ORDER.index(period) + 1, len(PERIOD_ORDER) - 1)]
        return longer if PERIOD_ORDER.index(longer) <= PERIOD_ORDER.index(self.max_fetch) else period


TIMEFRAMES: tuple[Timeframe, ...] = (
    Timeframe("5m", "5分足", "5m", (("1日", "1d"), ("5日", "5d"), ("1か月", "1mo")), "5d", (5, 25, 75), True, "1mo"),
    Timeframe("15m", "15分足", "15m", (("5日", "5d"), ("1か月", "1mo")), "1mo", (5, 25, 75), True, "1mo"),
    Timeframe(
        "1h", "1時間足", "1h", (("1か月", "1mo"), ("3か月", "3mo"), ("6か月", "6mo"), ("1年", "1y")), "3mo",
        (5, 25, 75), True, "2y",
    ),
    Timeframe(
        "1d", "日足", "1d", (("3か月", "3mo"), ("6か月", "6mo"), ("1年", "1y"), ("2年", "2y"), ("5年", "5y")), "6mo",
        (5, 25, 75), False, "max",
    ),
    Timeframe(
        "1wk", "週足", "1wk", (("1年", "1y"), ("2年", "2y"), ("5年", "5y"), ("10年", "10y"), ("全期間", "max")), "2y",
        (13, 26, 52), False, "max",
    ),
)


def get_timeframe(key: str) -> Timeframe:
    for timeframe in TIMEFRAMES:
        if timeframe.key == key:
            return timeframe
    raise ValueError(f"未対応の足です: {key}")


@dataclass(frozen=True)
class Quote:
    price: float
    as_of: date  # 価格の日付 (日足の日付)
    prev_close: float | None = None  # 前営業日の終値


class PriceSource(Protocol):
    def latest(self, inst: Instrument) -> Quote:
        """最新の価格 (取引時間中は現在値、時間外は直近の終値)。"""

    def on_date(self, inst: Instrument, day: date) -> Quote:
        """指定日の終値。休場日なら直前の営業日の終値。"""


class YFinanceSource:
    """Yahoo Finance の日足データを yfinance で取得する。

    日足の最後の行は取引時間中なら当日の途中経過 (現在値) になる。
    東証銘柄は Yahoo Finance 側でおよそ 20 分遅れのデータになる。
    """

    def __init__(self, cache_seconds: float = LATEST_CACHE_SECONDS) -> None:
        self.cache_seconds = cache_seconds
        self._latest_cache: dict[str, tuple[float, Quote]] = {}
        self._candle_cache: dict[tuple[str, str, str], tuple[float, pd.DataFrame]] = {}
        self._path_cache: dict[tuple[str, str, str], tuple[float, pd.DataFrame]] = {}

    def clear_cache(self) -> None:
        self._latest_cache.clear()
        self._candle_cache.clear()
        self._path_cache.clear()

    def latest(self, inst: Instrument) -> Quote:
        cached = self._latest_cache.get(inst.key)
        if cached is not None and time.monotonic() - cached[0] < self.cache_seconds:
            return cached[1]
        closes = self._daily_closes(inst, period="1mo")
        if not closes:
            raise PriceError(f"{inst.name}({inst.ticker}) の価格データを取得できませんでした。")
        day, price = closes[-1]
        prev_close = closes[-2][1] if len(closes) >= 2 else None
        quote = Quote(price, day, prev_close)
        self._latest_cache[inst.key] = (time.monotonic(), quote)
        return quote

    def on_date(self, inst: Instrument, day: date) -> Quote:
        start = day - timedelta(days=LOOKBACK_DAYS)
        end = day + timedelta(days=1)  # yfinance の end はその日を含まない
        closes = [
            (d, p)
            for d, p in self._daily_closes(inst, start=start.isoformat(), end=end.isoformat())
            if d <= day
        ]
        if not closes:
            raise PriceError(
                f"{day} 以前 {LOOKBACK_DAYS} 日間に {inst.name}({inst.ticker}) の価格データがありません。"
            )
        found_day, price = closes[-1]
        return Quote(price, found_day)

    def candles(self, inst: Instrument, timeframe: Timeframe, period: str) -> pd.DataFrame:
        """ローソク足 (Open/High/Low/Close/Volume)。

        移動平均線を計算できるよう、period より長めに取得する (表示する範囲は charts.visible_start で決める)。
        時刻はすべて日本時間 (タイムゾーン情報なし) にそろえる。日足・週足は日付 (0 時) にする。
        """
        fetch = timeframe.fetch_period(period)
        key = (inst.key, timeframe.key, fetch)
        cached = self._candle_cache.get(key)
        if cached is not None and time.monotonic() - cached[0] < self.cache_seconds:
            return cached[1]

        import pandas as pd

        df = self._history(inst, interval=timeframe.interval, period=fetch)
        columns = ["Open", "High", "Low", "Close", "Volume"]
        if df is None or df.empty or not set(columns) <= set(df.columns):
            raise PriceError(f"{inst.name}({inst.ticker}) の{timeframe.label}のデータがありません。期間を変えてみてください。")
        df = df[columns].dropna(subset=["Open", "High", "Low", "Close"]).copy()
        if timeframe.intraday:
            df.index = df.index.tz_convert("Asia/Tokyo").tz_localize(None)
        else:
            # 為替の日足はロンドン時間の 0 時なので、日本時間に直さずにその日付を使う
            df.index = pd.DatetimeIndex([pd.Timestamp(ts.date()) for ts in df.index])
        df = df[~df.index.duplicated(keep="last")]
        df["Volume"] = df["Volume"].fillna(0)
        if df.empty:
            raise PriceError(f"{inst.name}({inst.ticker}) の{timeframe.label}のデータがありません。期間を変えてみてください。")
        self._candle_cache[key] = (time.monotonic(), df)
        return df

    def path(self, inst: Instrument, since: datetime) -> pd.DataFrame:
        """since (日本時間) 以降に始まるローソク足 (Open/High/Low/Close)。決済注文の判定に使う。

        Yahoo の制限があるので、直近 58 日は 5 分足、720 日までは 1 時間足、それより前は日足を使う。
        使った足の種類は df.attrs["interval"] に入る。データが無ければ空の DataFrame を返す。
        """
        import pandas as pd

        now = datetime.now(timezone.utc).replace(tzinfo=None) + timedelta(hours=9)  # 日本時間
        age = now - since
        interval = "5m" if age <= timedelta(days=58) else "1h" if age <= timedelta(days=720) else "1d"
        start = (since - timedelta(days=1)).date().isoformat()
        key = (inst.key, interval, start)
        cached = self._path_cache.get(key)
        if cached is not None and time.monotonic() - cached[0] < self.cache_seconds:
            df = cached[1]
        else:
            end = (now + timedelta(days=2)).date().isoformat()
            df = self._history(inst, interval=interval, start=start, end=end)
            columns = ["Open", "High", "Low", "Close"]
            if df is None or df.empty or not set(columns) <= set(df.columns):
                df = pd.DataFrame(columns=columns, index=pd.DatetimeIndex([]))
            else:
                df = df[columns].dropna().copy()
                if interval == "1d":
                    df.index = pd.DatetimeIndex([pd.Timestamp(ts.date()) for ts in df.index])
                else:
                    df.index = df.index.tz_convert("Asia/Tokyo").tz_localize(None)
                df = df[~df.index.duplicated(keep="last")]
            self._path_cache[key] = (time.monotonic(), df)
        result = df[df.index >= pd.Timestamp(since)]
        result.attrs["interval"] = interval
        return result

    def _daily_closes(self, inst: Instrument, **kwargs) -> list[tuple[date, float]]:
        df = self._history(inst, interval="1d", **kwargs)
        if df is None or df.empty or "Close" not in df.columns:
            return []
        closes = df["Close"].dropna()
        return [(ts.date(), round(float(value), inst.price_decimals)) for ts, value in closes.items()]

    def _history(self, inst: Instrument, **kwargs) -> pd.DataFrame | None:
        try:
            import yfinance as yf
        except ImportError as e:
            raise PriceError(
                "yfinance がインストールされていません。`pip install -r requirements.txt` を実行してください。"
            ) from e

        # データが無いときの yfinance 自身のログは、こちらのエラーメッセージで代わりに伝える
        logging.getLogger("yfinance").setLevel(logging.CRITICAL)
        try:
            # auto_adjust=False: 配当で調整されていない、実際の価格を使う
            return yf.Ticker(inst.ticker).history(auto_adjust=False, actions=False, **kwargs)
        except Exception as e:  # 通信エラーやレート制限など、yfinance は色々な例外を投げる
            raise PriceError(f"{inst.name}({inst.ticker}) の価格取得に失敗しました: {e}") from e
''',
    'trade_sim/simulator.py': r'''"""シミュレーターの操作。コマンドライン (cli.py) と Colab の画面 (notebook.py) の共通部分。

各操作は表示用の Result を返し、表示のしかたは呼び出し側が決める。
"""

from __future__ import annotations

from collections.abc import Callable, Iterable, Sequence
from dataclasses import dataclass, field, replace
from datetime import date, datetime, timedelta, timezone
from pathlib import Path

from .instruments import (
    INSTRUMENTS,
    Instrument,
    UnknownInstrumentError,
    find_instrument,
    format_leverage,
    get_instrument,
)
from .portfolio import (
    LOSSCUT,
    REASON_LABELS,
    SIDE_LABELS,
    AccountStatus,
    Portfolio,
    PortfolioError,
    Position,
    SummaryRow,
    Trade,
    exit_prices,
    find_exit,
    parse_trade_time,
    required_margin,
)
from .prices import PriceError, PriceSource, YFinanceSource

# 日本に夏時間はないので、固定の +9 時間で日本時間になる
JST = timezone(timedelta(hours=9), "JST")


class InputError(Exception):
    """入力内容の誤り。"""


# 利用者に内容をそのまま伝えればよいエラー
USER_ERRORS = (InputError, PortfolioError, PriceError, UnknownInstrumentError)


def now_jst() -> datetime:
    """日本時間の現在時刻。Colab のように UTC で動く環境でも、取引時刻を日本時間で記録する。"""
    return datetime.now(JST).replace(tzinfo=None)


def parse_date(text: str | None) -> date | None:
    """"YYYY-MM-DD" を日付にする。空欄なら None。"""
    text = (text or "").strip()
    if not text:
        return None
    try:
        return date.fromisoformat(text.replace("/", "-"))
    except ValueError:
        raise InputError(f"日付は YYYY-MM-DD 形式で指定してください: {text}") from None


# ---- 表示用の整形 --------------------------------------------------------


def fmt_yen(value: float | None) -> str:
    if value is None:
        return "-"
    rounded = round(value)
    return "0円" if rounded == 0 else f"{rounded:+,}円"


def fmt_amount(value: float | None) -> str:
    """符号を付けない金額 (資金・証拠金など)。"""
    return "-" if value is None else f"{round(value):,}円"


def fmt_level(level: float | None) -> str:
    """証拠金維持率。建玉が無ければ "-"。"""
    return "-" if level is None else f"{level:,.1f}%"


def fmt_rate(rate: float | None) -> str:
    return "-" if rate is None else f"{rate * 100:.0f}%"


def fmt_change(inst: Instrument, price: float, prev_close: float | None) -> str:
    if not prev_close:
        return "-"
    diff = price - prev_close
    return f"{diff:+,.{inst.price_decimals}f} ({diff / prev_close * 100:+.2f}%)"


@dataclass
class Table:
    headers: list[str]
    rows: list[list[str]]
    aligns: str  # 列ごとの寄せ方 ("<" 左寄せ / ">" 右寄せ)
    footer: list[list[str]] = field(default_factory=list)


@dataclass
class Result:
    table: Table | None = None
    lines: list[str] = field(default_factory=list)  # 約定の結果や合計など
    notes: list[str] = field(default_factory=list)  # 注意書き (価格を取得できなかった等)
    ok: bool = True
    losscut: bool = False  # ロスカットが起きた


# ---- 操作 ----------------------------------------------------------------


class Simulator:
    def __init__(
        self,
        path: str | Path,
        source: PriceSource | None = None,
        clock: Callable[[], datetime] = now_jst,
    ) -> None:
        self.path = Path(path)
        self.source = source if source is not None else YFinanceSource()
        self.clock = clock

    def load(self) -> Portfolio:
        return Portfolio.load(self.path)

    def refresh_prices(self) -> None:
        """取得済みの価格を捨てて、次の操作で取得し直す。"""
        clear = getattr(self.source, "clear_cache", None)
        if clear is not None:
            clear()

    def quote(self, symbols: Iterable[str] = ()) -> Result:
        instruments = [find_instrument(s) for s in symbols] or list(INSTRUMENTS)
        rows, notes = [], []
        for inst in instruments:
            try:
                quote = self.source.latest(inst)
            except PriceError as e:
                rows.append([inst.name, inst.ticker, "-", "-", "-"])
                notes.append(str(e))
                continue
            rows.append(
                [
                    inst.name,
                    inst.ticker,
                    inst.format_price(quote.price),
                    fmt_change(inst, quote.price, quote.prev_close),
                    quote.as_of.isoformat(),
                ]
            )
        table = Table(["銘柄", "ティッカー", "現在値", "前日比", "日付"], rows, "<<>><")
        return Result(table=table, notes=notes, ok=len(notes) < len(instruments))

    def open(
        self,
        symbol: str,
        side: str,
        qty: int,
        price: float | None = None,
        day: date | None = None,
        leverage: float = 1.0,
        take_profit_yen: float | None = None,
        stop_loss_yen: float | None = None,
    ) -> Result:
        """新規にロング (side="long") またはショート (side="short") で建てる。

        必要証拠金 (取引金額 ÷ レバレッジ) が余力を超える注文はできない。
        take_profit_yen / stop_loss_yen (円) を渡すと、その金額の利益・損失になったら決済する注文も出す。
        """
        inst = find_instrument(symbol)
        if side not in SIDE_LABELS:
            raise InputError(f"売買区分が不正です: {side}")
        if isinstance(leverage, bool) or not isinstance(leverage, (int, float)) or not 1 <= leverage <= inst.max_leverage:
            raise InputError(
                f"{inst.name}のレバレッジは 1 倍から {format_leverage(inst.max_leverage)}までで指定してください: {leverage}"
            )
        if isinstance(qty, bool) or not isinstance(qty, int) or qty <= 0:
            raise InputError(f"数量は 1 以上の整数で指定してください: {qty}")
        for amount in (take_profit_yen, stop_loss_yen):
            if amount is not None and amount < 0:
                raise InputError(f"利益確定・損切りの金額は 0 以上で指定してください (0 なら注文なし): {amount}")
        events = self.check_triggers()
        if events is not None and events.losscut:
            events.notes.append("ロスカットが発生したため、この注文は出していません。")
            events.ok = False
            return events
        event_lines = events.lines if events else []

        price, at, note = self._resolve_price(inst, price, day)
        portfolio = self.load()
        status = self._status(portfolio)
        margin = required_margin(price, qty, leverage)
        if status.free_margin is None:
            raise InputError("現在値を取得できない建玉があるため、余力を計算できません。少し待ってからもう一度試してください。")
        if margin > status.free_margin:
            raise InputError(
                f"余力が足りません。この注文の必要証拠金は {fmt_amount(margin)}、余力は {fmt_amount(status.free_margin)}"
                " です。数量を減らすか、レバレッジを上げてください。"
            )
        position = portfolio.open(inst.key, side, qty, price, at, leverage, take_profit_yen, stop_loss_yen)
        portfolio.save(self.path)
        lines = [
            *event_lines,
            f"新規{SIDE_LABELS[side]}: #{position.id} {inst.name} {inst.format_qty(position.qty)}"
            f" @ {inst.format_price(price)}円 ({note})"
            f"  レバレッジ {format_leverage(position.leverage)}・必要証拠金 {fmt_amount(position.required_margin)}",
        ]
        if position.take_profit is not None or position.stop_loss is not None:
            lines.append(_exit_line(inst, position))
        return Result(lines=lines)

    def set_exit(
        self, position_id: int | str, take_profit_yen: float | None = None, stop_loss_yen: float | None = None
    ) -> Result:
        """保有中の建玉に決済注文を出す (出し直す)。

        金額 (円) が None ならその注文は今のまま、0 なら取り消す。
        今の価格ですでに届いている金額は指定できない。注文を出した時点より後の値動きで判定する。
        """
        for amount in (take_profit_yen, stop_loss_yen):
            if amount is not None and amount < 0:
                raise InputError(f"利益確定・損切りの金額は 0 以上で指定してください (0 なら注文なし): {amount}")
        events = self.check_triggers()
        event_lines = events.lines if events else []
        portfolio = self.load()
        position_id = _parse_position_id(position_id)
        try:
            position = portfolio.find(position_id)
        except PortfolioError:
            if events is not None:  # 決済注文やロスカットで、ちょうど決済されたところ
                events.notes.append(f"建玉 #{position_id} はすでに決済されています。")
                return events
            raise
        inst = get_instrument(position.symbol)
        new_tp, new_sl = exit_prices(position, take_profit_yen, stop_loss_yen)
        tp = position.take_profit if take_profit_yen is None else new_tp
        sl = position.stop_loss if stop_loss_yen is None else new_sl
        if tp is not None or sl is not None:
            latest = self.source.latest(inst).price
            now = self.clock()
            reached = [
                name
                for name, probe in (
                    ("損切り", replace(position, take_profit=None, stop_loss=sl)),
                    ("利益確定", replace(position, take_profit=tp, stop_loss=None)),
                )
                if find_exit(probe, (), latest, now) is not None
            ]
            if reached:
                raise InputError(
                    f"今の価格 {inst.format_price(latest)} で、すでに{reached[0]}の価格に届いています。金額を変えてください。"
                )
        position = portfolio.set_exit(position_id, tp, sl, self.clock().strftime("%Y-%m-%d %H:%M"))
        portfolio.save(self.path)
        if tp is None and sl is None:
            return Result(lines=[*event_lines, f"決済注文を取り消しました: #{position.id} {inst.name}"])
        return Result(lines=[*event_lines, _exit_line(inst, position)])

    def close(
        self,
        ids: Sequence[int | str],
        qty: int | None = None,
        price: float | None = None,
        day: date | None = None,
    ) -> Result:
        """建玉を決済する。ids に "all" を渡すとすべての建玉。qty を指定すると一部だけ決済する。"""
        events = self.check_triggers()
        if events is not None and events.losscut:
            events.notes.append("ロスカットで、保有中の建玉はすべて決済済みです。")
            return events
        event_lines = events.lines if events else []
        portfolio = self.load()
        if [str(i).lower() for i in ids] == ["all"]:
            targets = [p.id for p in portfolio.positions]
            if not targets:
                return Result(lines=[*event_lines, "決済する建玉がありません。"])
        else:
            targets = list(dict.fromkeys(_parse_position_id(i) for i in ids))
            closed = [i for i in targets if i not in {p.id for p in portfolio.positions}]
            if closed and events is not None:  # 決済注文で、ちょうど決済されたところ
                events.notes.append(f"建玉 #{closed[0]} は決済注文ですでに決済されています。")
                return events
        if not targets:
            raise InputError("決済する建玉の ID を指定してください。")
        if len(targets) > 1 and qty is not None:
            raise InputError("一部決済 (数量の指定) は、建玉を 1 つだけ指定したときに使えます。")
        if len(targets) > 1 and price is not None:
            raise InputError("価格の指定は、建玉を 1 つだけ指定したときに使えます。")

        # 先にすべての決済価格を決めてから決済する (途中で失敗したら何も保存しない)
        orders = []
        for position_id in targets:
            inst = get_instrument(portfolio.find(position_id).symbol)
            orders.append((position_id, inst, *self._resolve_price(inst, price, day)))

        lines = []
        for position_id, inst, exit_price, at, note in orders:
            trade = portfolio.close(position_id, exit_price, at, qty)
            lines.append(_close_line(inst, trade, note))
        portfolio.save(self.path)
        lines.append(f"実現損益の通算: {fmt_yen(portfolio.realized_pnl)}")
        return Result(lines=[*event_lines, *lines])

    def positions(self) -> Result:
        events = self.check_triggers()
        event_lines = events.lines if events else []
        portfolio = self.load()
        if not portfolio.positions:
            return Result(lines=[*event_lines, "保有中の建玉はありません。"])
        prices, notes = self.latest_prices({p.symbol for p in portfolio.positions})
        rows = []
        total = 0.0
        for position in portfolio.positions:
            inst = get_instrument(position.symbol)
            price = prices.get(position.symbol)
            if price is None:
                current, pnl_text = "-", "-"
            else:
                pnl = position.unrealized_pnl(price)
                total += pnl
                current, pnl_text = inst.format_price(price), fmt_yen(pnl)
            rows.append(
                [
                    f"#{position.id}",
                    inst.name,
                    SIDE_LABELS[position.side],
                    inst.format_qty(position.qty),
                    format_leverage(position.leverage),
                    inst.format_price(position.entry_price),
                    current,
                    pnl_text,
                    fmt_amount(position.required_margin),
                    "-" if position.take_profit is None else inst.format_price(position.take_profit),
                    "-" if position.stop_loss is None else inst.format_price(position.stop_loss),
                    position.opened_at,
                ]
            )
        headers = [
            "建玉", "銘柄", "売買", "数量", "レバレッジ", "建値", "現在値", "評価損益", "必要証拠金", "利確", "損切り", "約定日時",
        ]
        suffix = " (現在値を取得できなかった建玉を除く)" if notes else ""
        used = sum(p.required_margin for p in portfolio.positions)
        return Result(
            table=Table(headers, rows, "<<<>>>>>>>><"),
            lines=[*event_lines, f"評価損益の合計: {fmt_yen(total)}{suffix}　必要証拠金の合計: {fmt_amount(used)}"],
            notes=notes,
        )

    def history(self, symbol: str | None = None) -> Result:
        trades = self.load().trades
        if symbol:
            key = find_instrument(symbol).key
            trades = [t for t in trades if t.symbol == key]
        if not trades:
            return Result(lines=["決済済みの取引はまだありません。"])
        rows = []
        for trade in trades:
            inst = get_instrument(trade.symbol)
            rows.append(
                [
                    f"#{trade.position_id}",
                    inst.name,
                    SIDE_LABELS[trade.side],
                    inst.format_qty(trade.qty),
                    format_leverage(trade.leverage),
                    inst.format_price(trade.entry_price),
                    inst.format_price(trade.exit_price),
                    fmt_yen(trade.pnl),
                    trade.opened_at,
                    trade.closed_at,
                    REASON_LABELS.get(trade.reason, ""),
                ]
            )
        headers = ["建玉", "銘柄", "売買", "数量", "レバレッジ", "建値", "決済値", "損益", "約定日時", "決済日時", "備考"]
        return Result(
            table=Table(headers, rows, "<<<>>>>><<<"),
            lines=[f"実現損益の合計: {fmt_yen(sum(t.pnl for t in trades))} ({len(trades)} 件)"],
        )

    def summary_rows(self) -> tuple[list[SummaryRow], list[str]]:
        """銘柄別の損益 (数値) と、現在値を取得できなかった銘柄のエラー。"""
        portfolio = self.load()
        prices, errors = self.latest_prices({p.symbol for p in portfolio.positions})
        return portfolio.summary(prices), errors

    def summary(self) -> Result:
        """銘柄別と通算の損益。通算損益 = 実現損益 + 評価損益。口座の状態 (証拠金) も表示する。"""
        events = self.check_triggers()
        rows, errors = self.summary_rows()
        status, _ = self.account_status()

        table_rows = [
            [
                get_instrument(r.symbol).name,
                str(r.trades),
                fmt_rate(r.win_rate),
                fmt_yen(r.realized),
                str(r.open_positions),
                fmt_yen(r.unrealized),
                fmt_yen(r.total),
            ]
            for r in rows
        ]
        trades = sum(r.trades for r in rows)
        wins = sum(r.wins for r in rows)
        realized = round(sum(r.realized for r in rows), 2)
        unrealized = round(sum(r.unrealized for r in rows if r.unrealized is not None), 2)
        missing = any(r.unrealized is None for r in rows)
        total = round(realized + unrealized, 2)
        footer = [
            [
                "合計",
                str(trades),
                fmt_rate(wins / trades if trades else None),
                fmt_yen(realized),
                str(sum(r.open_positions for r in rows)),
                "-" if missing else fmt_yen(unrealized),
                "-" if missing else fmt_yen(total),
            ]
        ]
        headers = ["銘柄", "決済回数", "勝率", "実現損益", "建玉数", "評価損益", "合計"]
        notes = []
        if missing:
            notes = ["現在値を取得できなかった銘柄の評価損益は、通算損益に含まれていません。", *errors]
        return Result(
            table=Table(headers, table_rows, "<>>>>>>", footer),
            lines=[
                *(events.lines if events else []),
                f"通算損益: {fmt_yen(total)}  (実現損益 {fmt_yen(realized)} + 評価損益 {fmt_yen(unrealized)})",
                *_account_lines(status),
            ],
            notes=notes,
        )

    # ---- 口座 ------------------------------------------------------------

    def account_status(self) -> tuple[AccountStatus, list[str]]:
        """口座の状態 (資金・有効証拠金・余力・証拠金維持率) と、現在値を取得できなかった銘柄のエラー。"""
        portfolio = self.load()
        prices, errors = self.latest_prices({p.symbol for p in portfolio.positions})
        return portfolio.status(prices), errors

    def account(self) -> Result:
        events = self.check_triggers()
        status, errors = self.account_status()
        return Result(lines=[*(events.lines if events else []), *_account_lines(status)], notes=errors)

    def set_account(self, initial_capital: float | None = None, losscut_level: float | None = None) -> Result:
        """初期資金とロスカット水準 (%、0 ならロスカットしない) を変える。"""
        portfolio = self.load()
        portfolio.set_account(initial_capital, losscut_level)
        portfolio.save(self.path)
        lines = []
        if initial_capital is not None:
            lines.append(f"初期資金を {fmt_amount(portfolio.initial_capital)} にしました。")
        if losscut_level is not None:
            lines.append(f"ロスカット水準を {fmt_losscut(portfolio.losscut_level)} にしました。")
        return Result(lines=lines)

    def check_triggers(self) -> Result | None:
        """決済注文 (利益確定・損切り) とロスカットを判定して、条件を満たした建玉を決済する。

        何も起きなければ None。ロスカットが起きたら Result.losscut が True。
        """
        exits = self.check_exit_orders()
        losscut = self.check_losscut()
        if exits is None and losscut is None:
            return None
        lines = [*(exits.lines if exits else []), *(losscut.lines if losscut else [])]
        return Result(lines=lines, losscut=losscut is not None)

    def check_exit_orders(self) -> Result | None:
        """決済注文を出した後の値動き (ローソク足) を順に見て、価格に届いた建玉を、届いた時点で決済する。

        アプリを閉じていた間の値動きも、ローソク足をさかのぼって判定する。
        """
        portfolio = self.load()
        targets = [p for p in portfolio.positions if p.take_profit is not None or p.stop_loss is not None]
        if not targets:
            return None
        now = self.clock()
        fills = []  # (時刻, 記録する日時, 建玉, 約定価格, 理由)
        get_path = getattr(self.source, "path", None)
        for inst in INSTRUMENTS:
            positions = [p for p in targets if p.symbol == inst.key]
            if not positions:
                continue
            starts = {p.id: parse_trade_time(p.exit_set_at or p.opened_at) for p in positions}
            candles: list[tuple[datetime, float, float, float]] = []
            daily = False
            if get_path is not None:
                try:
                    path = get_path(inst, min(starts.values()))
                except PriceError:
                    path = None
                if path is not None and len(path):
                    daily = path.attrs.get("interval") == "1d"
                    candles = [
                        (ts.to_pydatetime(), float(o), float(h), float(low))
                        for ts, o, h, low in zip(path.index, path["Open"], path["High"], path["Low"])
                    ]
            try:
                latest = self.source.latest(inst).price
            except PriceError:
                latest = None
            for position in positions:
                after = [c for c in candles if c[0] >= starts[position.id]]
                hit = find_exit(position, after, latest, now)
                if hit is None:
                    continue
                when, price, reason = hit
                at = when.strftime("%Y-%m-%d" if daily and when != now else "%Y-%m-%d %H:%M")
                fills.append((when, at, position, round(price, inst.price_decimals + 2), reason))
        if not fills:
            return None
        fills.sort(key=lambda fill: fill[0])
        lines = []
        for _, at, position, price, reason in fills:
            inst = get_instrument(position.symbol)
            trade = portfolio.close(position.id, price, at, reason=reason)
            lines.append(_close_line(inst, trade, f"{REASON_LABELS[reason]} {at}"))
        portfolio.save(self.path)
        lines.append(f"実現損益の通算: {fmt_yen(portfolio.realized_pnl)}")
        return Result(lines=lines)

    def check_losscut(self) -> Result | None:
        """証拠金維持率がロスカット水準を下回っていたら、すべての建玉を今の価格で決済する。

        価格を確認したとき (注文・決済・表示のたび) に判定する。アプリを閉じている間は判定しないので、
        次に開いたときの価格で決済される。
        """
        portfolio = self.load()
        if not portfolio.positions or portfolio.losscut_level <= 0:
            return None
        prices, _ = self.latest_prices({p.symbol for p in portfolio.positions})
        status = portfolio.status(prices)
        if not status.losscut_triggered:
            return None
        at = self.clock().strftime("%Y-%m-%d %H:%M")
        lines = [
            f"ロスカット: 証拠金維持率が {fmt_level(status.margin_level)} になり、ロスカット水準 "
            f"{fmt_losscut(status.losscut_level)} を下回ったため、すべての建玉を決済しました。"
        ]
        for position in list(portfolio.positions):
            inst = get_instrument(position.symbol)
            trade = portfolio.close(position.id, prices[position.symbol], at, reason=LOSSCUT)
            lines.append(_close_line(inst, trade, "ロスカット"))
        portfolio.save(self.path)
        lines.append(f"実現損益の通算: {fmt_yen(portfolio.realized_pnl)}")
        return Result(lines=lines)

    def reset(self) -> Result:
        """建玉と取引履歴をすべて削除する。"""
        if not self.path.exists():
            return Result(lines=[f"データファイル {self.path} はありません。"])
        self.path.unlink()
        return Result(lines=["データを削除しました。"])

    # ---- 内部 ------------------------------------------------------------

    def _resolve_price(self, inst: Instrument, price: float | None, day: date | None) -> tuple[float, str, str]:
        """約定価格を決める。戻り値は (価格, 記録する約定日時, 説明)。"""
        now = self.clock()
        if day is not None and day > now.date():
            raise InputError(f"未来の日付 ({day}) は指定できません。")
        if price is not None:
            if not price > 0:
                raise InputError(f"価格は 0 より大きい値を指定してください: {price}")
            at = day.isoformat() if day else now.strftime("%Y-%m-%d %H:%M")
            return price, at, "指定価格"
        if day is not None:
            quote = self.source.on_date(inst, day)
            if quote.as_of == now.date():
                note = f"当日 {quote.as_of} の最新値"  # 取引時間中なら確定した終値ではない
            elif quote.as_of != day:
                note = f"{day} は取引がないため、直前の営業日 {quote.as_of} の終値"
            else:
                note = f"{quote.as_of} の終値"
            return quote.price, quote.as_of.isoformat(), note
        quote = self.source.latest(inst)
        return quote.price, now.strftime("%Y-%m-%d %H:%M"), f"{quote.as_of} 時点の最新値"

    def _status(self, portfolio: Portfolio) -> AccountStatus:
        prices, _ = self.latest_prices({p.symbol for p in portfolio.positions})
        return portfolio.status(prices)

    def latest_prices(self, symbols: set[str]) -> tuple[dict[str, float | None], list[str]]:
        """保有銘柄の現在値をまとめて取得する。取得できなかった銘柄は None。"""
        prices: dict[str, float | None] = {}
        errors = []
        for inst in INSTRUMENTS:
            if inst.key not in symbols:
                continue
            try:
                prices[inst.key] = self.source.latest(inst).price
            except PriceError as e:
                prices[inst.key] = None
                errors.append(str(e))
        return prices, errors


def _close_line(inst: Instrument, trade: Trade, note: str) -> str:
    return (
        f"決済: #{trade.position_id} {inst.name} {SIDE_LABELS[trade.side]} {inst.format_qty(trade.qty)}"
        f"  建値 {inst.format_price(trade.entry_price)} → 決済値 {inst.format_price(trade.exit_price)}"
        f" ({note})  損益 {fmt_yen(trade.pnl)}"
    )


def _exit_line(inst: Instrument, position: Position) -> str:
    parts = []
    if position.take_profit is not None:
        parts.append(
            f"利確 {inst.format_price(position.take_profit)} ({fmt_yen(position.exit_pnl(position.take_profit))})"
        )
    if position.stop_loss is not None:
        parts.append(f"損切り {inst.format_price(position.stop_loss)} ({fmt_yen(position.exit_pnl(position.stop_loss))})")
    return f"決済注文: #{position.id} {inst.name} {SIDE_LABELS[position.side]}　" + "・".join(parts)


def fmt_losscut(level: float) -> str:
    return "なし (ロスカットしない)" if level <= 0 else f"{level:g}%"


def _account_lines(status: AccountStatus) -> list[str]:
    return [
        f"有効証拠金: {fmt_amount(status.equity)}  (資金 {fmt_amount(status.cash)} + 評価損益 {fmt_yen(status.unrealized)})"
        f"　初期資金: {fmt_amount(status.initial_capital)}",
        f"必要証拠金: {fmt_amount(status.used_margin)}　余力: {fmt_amount(status.free_margin)}"
        f"　証拠金維持率: {fmt_level(status.margin_level)}　ロスカット水準: {fmt_losscut(status.losscut_level)}",
    ]


def _parse_position_id(value: int | str) -> int:
    if isinstance(value, int) and not isinstance(value, bool):
        return value
    try:
        return int(str(value).strip().lstrip("#"))
    except ValueError:
        raise InputError(f"建玉 ID は数字で指定してください: {value}") from None
''',
    'trade_sim/table.py': r'''"""全角文字を含む表を桁をそろえて表示する。"""

from __future__ import annotations

import unicodedata
from collections.abc import Sequence


def text_width(text: str) -> int:
    """端末上での表示幅 (全角は 2、半角は 1)。"""
    return sum(2 if unicodedata.east_asian_width(ch) in ("W", "F") else 1 for ch in text)


def _pad(text: str, width: int, align: str) -> str:
    gap = " " * (width - text_width(text))
    return gap + text if align == ">" else text + gap


def render_table(
    headers: Sequence[str],
    rows: Sequence[Sequence[str]],
    aligns: str,
    footer: Sequence[Sequence[str]] = (),
) -> str:
    """aligns は列ごとの寄せ方 ("<" 左寄せ / ">" 右寄せ) を並べた文字列。"""
    all_rows = [headers, *rows, *footer]
    widths = [max(text_width(row[i]) for row in all_rows) for i in range(len(headers))]

    def line(cells: Sequence[str]) -> str:
        return "  ".join(_pad(c, w, a) for c, w, a in zip(cells, widths, aligns)).rstrip()

    separator = "  ".join("-" * w for w in widths)
    lines = [line(headers), separator, *(line(r) for r in rows)]
    if footer:
        lines += [separator, *(line(r) for r in footer)]
    return "\n".join(lines)
''',
    'trade_sim/webapp.py': r'''"""ブラウザで使う取引シミュレーター (Streamlit)。

起動:  streamlit run app.py
"""

from __future__ import annotations

import html
import math
import os
from collections.abc import Callable
from datetime import date

import pandas as pd
import streamlit as st

from .charts import DOWN, UP, candle_figure, equity_figure, pnl_by_instrument_figure
from .instruments import INSTRUMENTS, LEVERAGE_CHOICES, Instrument, format_leverage, get_instrument
from .portfolio import (
    LONG,
    LOSSCUT,
    REASON_LABELS,
    SHORT,
    SIDE_LABELS,
    AccountStatus,
    Portfolio,
    Position,
    SummaryRow,
    exit_prices,
)
from .prices import TIMEFRAMES, PriceError, YFinanceSource, get_timeframe
from .simulator import (
    USER_ERRORS,
    Result,
    Simulator,
    fmt_amount,
    fmt_change,
    fmt_level,
    fmt_losscut,
    fmt_rate,
    fmt_yen,
)

DEFAULT_FILE = os.environ.get("TRADE_SIM_FILE", "portfolio.json")
DEFAULT_QTY = {"stock": 100, "fx": 10000}
QTY_STEP = {"stock": 100, "fx": 1000}
EXIT_STEP = 1000  # 利益確定・損切りの金額 (円) の刻み
REFRESH_CHOICES = {"30秒": 30, "1分": 60, "5分": 300}
SIDE_CHOICES = {LONG: "ロング（買い）", SHORT: "ショート（売り）"}
OLDEST_DATE = date(2000, 1, 1)
WARN = "#d97706"  # 証拠金維持率が低いときの色


# ---- 状態 ----------------------------------------------------------------


@st.cache_resource
def _shared_source() -> YFinanceSource:
    """価格の取得元。取得済みの価格を使い回すため、アプリ全体で 1 つにする。"""
    return YFinanceSource()


def _simulator() -> Simulator:
    # テストでは session_state["price_source"] にテスト用の価格ソースを入れる
    source = st.session_state.get("price_source") or _shared_source()
    return Simulator(st.session_state.get("data_file") or DEFAULT_FILE, source)


def _name(key: str) -> str:
    return get_instrument(key).name


# ---- ボタンの処理 (コールバック) ------------------------------------------
# コールバックは画面を描き直す前に実行されるので、画面全体に操作の結果が反映される。


def _run_action(action: Callable[[], Result]) -> None:
    st.session_state.pop("notice", None)  # 前のロスカット・決済注文の知らせは、次の操作で消す
    try:
        result = action()
    except USER_ERRORS as e:
        st.session_state["flash"] = {"kind": "error", "text": str(e), "toast": False}
        return
    text = "\n".join([*result.lines, *(f"※ {note}" for note in result.notes)])
    st.session_state["flash"] = {"kind": "success" if result.ok else "error", "text": text, "toast": result.ok}


def _place_order() -> None:
    state = st.session_state
    inst = get_instrument(state["symbol"])
    sim = _simulator()
    _run_action(
        lambda: sim.open(
            inst.key,
            state["order_side"],
            int(state[f"order_qty_{inst.kind}"]),
            day=state.get("order_date"),
            leverage=float(state[f"order_leverage_{inst.kind}"]),
            take_profit_yen=float(state.get("order_tp") or 0) or None,
            stop_loss_yen=float(state.get("order_sl") or 0) or None,
        )
    )


def _set_exit() -> None:
    state = st.session_state
    sim = _simulator()
    # 0 は取り消し。両方を毎回指定する
    _run_action(
        lambda: sim.set_exit(
            state["close_id"], take_profit_yen=float(state["exit_tp"] or 0), stop_loss_yen=float(state["exit_sl"] or 0)
        )
    )
    state.pop("exit_loaded_for", None)  # 設定した内容で入力欄を読み込み直す


def _close_selected() -> None:
    state = st.session_state
    sim = _simulator()
    qty = int(state.get("close_qty") or 0) or None  # 0 ならすべて決済
    _run_action(lambda: sim.close([state["close_id"]], qty=qty, day=state.get("close_date")))
    state["close_qty"] = 0


def _close_all() -> None:
    state = st.session_state
    sim = _simulator()
    _run_action(lambda: sim.close(["all"], day=state.get("close_date")))


def _save_account() -> None:
    state = st.session_state
    sim = _simulator()
    _run_action(
        lambda: sim.set_account(
            initial_capital=float(state["account_capital"]), losscut_level=float(state["account_losscut"])
        )
    )


def _refresh_prices() -> None:
    # 前の操作のメッセージは消す
    st.session_state.pop("flash", None)
    st.session_state.pop("notice", None)
    _simulator().refresh_prices()


# ---- 表 ------------------------------------------------------------------


def _pnl_color(value: object) -> str:
    if value is None or pd.isna(value):
        return ""
    if value > 0:
        return f"color: {UP}"
    if value < 0:
        return f"color: {DOWN}"
    return ""


def _show_table(df: pd.DataFrame, pnl_columns: tuple[str, ...] = (), right_columns: tuple[str, ...] = ()) -> None:
    """損益の列は赤 (プラス) / 青 (マイナス) で表示し、数字の列は右寄せにする。"""
    styler = df.style.format({c: fmt_yen for c in pnl_columns}, na_rep="-")
    if pnl_columns:
        styler = styler.map(_pnl_color, subset=list(pnl_columns))
    config = {c: st.column_config.TextColumn(alignment="right") for c in right_columns}
    config.update({c: st.column_config.NumberColumn(alignment="right") for c in pnl_columns})
    st.dataframe(styler, hide_index=True, column_config=config)


def _exit_text(inst: Instrument, position: Position, price: float | None) -> str:
    if price is None:
        return "-"
    return f"{inst.format_price(price)}（{fmt_yen(position.exit_pnl(price))}）"


def _exit_amounts(position: Position) -> tuple[int, int]:
    """建玉の決済注文を、利益確定・損切りの金額 (円) に直す。注文が無ければ 0。"""
    tp = round(position.exit_pnl(position.take_profit)) if position.take_profit is not None else 0
    sl = -round(position.exit_pnl(position.stop_loss)) if position.stop_loss is not None else 0
    return tp, sl


def _positions_frame(portfolio: Portfolio, prices: dict[str, float | None]) -> pd.DataFrame:
    rows = []
    for p in portfolio.positions:
        inst = get_instrument(p.symbol)
        price = prices.get(p.symbol)
        rows.append(
            {
                "建玉": f"#{p.id}",
                "銘柄": inst.name,
                "売買": SIDE_LABELS[p.side],
                "数量": inst.format_qty(p.qty),
                "レバレッジ": format_leverage(p.leverage),
                "建値": inst.format_price(p.entry_price),
                "現在値": inst.format_price(price) if price is not None else "-",
                "評価損益": p.unrealized_pnl(price) if price is not None else None,
                "必要証拠金": fmt_amount(p.required_margin),
                "利確": _exit_text(inst, p, p.take_profit),
                "損切り": _exit_text(inst, p, p.stop_loss),
                "約定日時": p.opened_at,
            }
        )
    return pd.DataFrame(rows)


def _summary_frame(rows: list[SummaryRow]) -> pd.DataFrame:
    records = [
        {
            "銘柄": get_instrument(r.symbol).name,
            "決済回数": r.trades,
            "勝率": fmt_rate(r.win_rate),
            "実現損益": r.realized,
            "建玉数": r.open_positions,
            "評価損益": r.unrealized,
            "合計": r.total,
        }
        for r in rows
    ]
    trades = sum(r.trades for r in rows)
    missing = any(r.unrealized is None for r in rows)
    unrealized = sum(r.unrealized for r in rows if r.unrealized is not None)
    realized = sum(r.realized for r in rows)
    records.append(
        {
            "銘柄": "合計",
            "決済回数": trades,
            "勝率": fmt_rate(sum(r.wins for r in rows) / trades if trades else None),
            "実現損益": realized,
            "建玉数": sum(r.open_positions for r in rows),
            "評価損益": None if missing else unrealized,
            "合計": None if missing else realized + unrealized,
        }
    )
    return pd.DataFrame(records)


def _history_frame(portfolio: Portfolio) -> pd.DataFrame:
    rows = []
    for t in reversed(portfolio.trades):  # 新しい順
        inst = get_instrument(t.symbol)
        rows.append(
            {
                "建玉": f"#{t.position_id}",
                "銘柄": inst.name,
                "売買": SIDE_LABELS[t.side],
                "数量": inst.format_qty(t.qty),
                "レバレッジ": format_leverage(t.leverage),
                "建値": inst.format_price(t.entry_price),
                "決済値": inst.format_price(t.exit_price),
                "損益": t.pnl,
                "約定日時": t.opened_at,
                "決済日時": t.closed_at,
                "備考": REASON_LABELS.get(t.reason, ""),
            }
        )
    return pd.DataFrame(rows)


# ---- 画面の部品 ------------------------------------------------------------


def _kpi(
    column: st.delta_generator.DeltaGenerator, label: str, text: str, note: str, help: str, color: str | None = None
) -> None:
    """大きな数字のカード。どれも同じ構成にして高さをそろえる。"""
    style = "font-size: clamp(1.15rem, 2vw, 1.75rem); font-weight: 700; line-height: 1.3; white-space: nowrap;"
    if color:
        style += f" color: {color};"
    with column.container(border=True):
        st.caption(label, help=help)
        st.markdown(f'<div style="{style}">{html.escape(text)}</div>', unsafe_allow_html=True)
        st.caption(note)


def _pnl_color_of(value: float | None) -> str | None:
    if value is None or round(value) == 0:
        return None
    return UP if value > 0 else DOWN


def _account_header(status: AccountStatus, errors: list[str]) -> None:
    total = status.realized + (status.unrealized or 0)
    level = status.margin_level
    level_color = None
    if level is not None and status.losscut_level > 0 and level < status.losscut_level * 2:
        level_color = WARN  # ロスカット水準の 2 倍を下回ったら注意の色にする

    columns = st.columns(4)
    _kpi(
        columns[0], "有効証拠金", fmt_amount(status.equity), f"初期資金 {fmt_amount(status.initial_capital)}",
        "資金（初期資金 + 実現損益）に評価損益を足した金額",
    )
    _kpi(
        columns[1], "通算損益", fmt_yen(total), f"実現 {fmt_yen(status.realized)}・評価 {fmt_yen(status.unrealized)}",
        "決済済みの損益（実現損益）と、保有中の建玉の含み損益（評価損益）の合計", _pnl_color_of(total),
    )
    _kpi(
        columns[2], "余力", fmt_amount(status.free_margin), f"必要証拠金 {fmt_amount(status.used_margin)}",
        "新しく建てるのに使える証拠金（有効証拠金 − 必要証拠金）",
    )
    _kpi(
        columns[3], "証拠金維持率", fmt_level(level), f"ロスカット水準 {fmt_losscut(status.losscut_level)}",
        "有効証拠金 ÷ 必要証拠金。ロスカット水準を下回ると、すべての建玉を決済します", level_color,
    )
    if errors:
        st.warning("現在値を取得できなかった銘柄があります。評価損益・余力が正しくない可能性があります。  \n" + "  \n".join(errors))


def _quotes(sim: Simulator) -> None:
    columns = st.columns(len(INSTRUMENTS))
    for column, inst in zip(columns, INSTRUMENTS):
        try:
            quote = sim.source.latest(inst)
        except PriceError as e:
            column.metric(inst.name, "-", help=str(e), border=True)
            continue
        diff = quote.price - quote.prev_close if quote.prev_close else 0
        column.metric(
            inst.name,
            inst.format_price(quote.price),
            delta=fmt_change(inst, quote.price, quote.prev_close) if quote.prev_close else None,
            delta_color="red" if diff > 0 else "blue" if diff < 0 else "off",
            help=f"{inst.ticker}　{quote.as_of} 時点",
            border=True,
        )


def _show_flash() -> None:
    flash = st.session_state.get("flash")
    if not flash:
        return
    text = flash["text"].replace("\n", "  \n")
    if flash["kind"] == "error":
        st.error(text, icon=":material/error:")
    else:
        st.success(text, icon=":material/check_circle:")
        if flash["toast"]:
            st.toast(flash["text"].splitlines()[0], icon=":material/check_circle:")
            flash["toast"] = False


def _estimate_html(rows: list[tuple[str, str]]) -> str:
    cells = "".join(
        f'<div style="opacity: 0.75;">{html.escape(label)}</div>'
        f'<div style="text-align: right; font-variant-numeric: tabular-nums;">{html.escape(value)}</div>'
        for label, value in rows
    )
    return f'<div style="display: grid; grid-template-columns: auto auto; gap: 2px 12px; font-size: 0.9rem;">{cells}</div>'


# ---- 取引タブ --------------------------------------------------------------


def _order_panel(sim: Simulator, inst: Instrument, status: AccountStatus, today: date) -> None:
    """チャートの横に置く注文パネル。チャートで選んでいる銘柄を注文する。"""
    state = st.session_state
    kind = inst.kind
    qty_key, leverage_key = f"order_qty_{kind}", f"order_leverage_{kind}"
    state.setdefault(qty_key, DEFAULT_QTY[kind])
    state.setdefault(leverage_key, 1.0)
    state.setdefault("order_side", LONG)
    state.setdefault("order_tp", 0)
    state.setdefault("order_sl", 0)

    with st.container(border=True):
        st.markdown(f"**{inst.name}を注文**")
        try:
            quote = sim.source.latest(inst)
        except PriceError as e:
            quote = None
            st.warning(str(e))
        if quote is not None:
            diff = quote.price - quote.prev_close if quote.prev_close else 0
            st.metric(
                "現在値",
                f"{inst.format_price(quote.price)}円",
                delta=fmt_change(inst, quote.price, quote.prev_close) if quote.prev_close else None,
                delta_color="red" if diff > 0 else "blue" if diff < 0 else "off",
                help=f"{quote.as_of} 時点",
            )
        st.segmented_control(
            "売買", list(SIDE_CHOICES), format_func=SIDE_CHOICES.get, required=True, key="order_side", width="stretch"
        )
        st.number_input(f"数量（{inst.unit}）", min_value=1, step=QTY_STEP[kind], key=qty_key)
        st.select_slider(
            f"レバレッジ（{format_leverage(inst.max_leverage)}まで）",
            options=list(LEVERAGE_CHOICES[kind]),
            format_func=format_leverage,
            key=leverage_key,
        )
        st.date_input(
            "約定日（空欄なら最新の価格）",
            value=None,
            min_value=OLDEST_DATE,
            max_value=today,
            key="order_date",
            help="日付を入れると、その日の終値で約定します（休場日なら直前の営業日）。",
        )
        c1, c2 = st.columns(2)
        c1.number_input(
            "利益確定（円）",
            min_value=0,
            step=EXIT_STEP,
            key="order_tp",
            help="この金額の利益になったら決済します。0 なら注文しません。",
        )
        c2.number_input(
            "損切り（円）",
            min_value=0,
            step=EXIT_STEP,
            key="order_sl",
            help="この金額の損失になったら決済します。0 なら注文しません。",
        )

        if quote is not None:
            qty, leverage = int(state[qty_key]), float(state[leverage_key])
            notional = quote.price * qty
            margin = notional / leverage
            free = status.free_margin
            rows = [("取引金額", fmt_amount(notional)), ("必要証拠金", fmt_amount(margin)), ("余力", fmt_amount(free))]
            if free is not None:
                step = QTY_STEP[kind]
                max_qty = max(0, math.floor(free * leverage / quote.price / step) * step)
                rows.append(("建てられる最大数量", inst.format_qty(max_qty)))
            if state.get("order_tp") or state.get("order_sl"):
                probe = Position(0, inst.key, state["order_side"], qty, quote.price, "")
                try:
                    tp, sl = exit_prices(probe, state.get("order_tp") or None, state.get("order_sl") or None)
                except USER_ERRORS as e:
                    st.warning(str(e))
                else:
                    if tp is not None:
                        rows.append(("利確の価格", inst.format_price(tp)))
                    if sl is not None:
                        rows.append(("損切りの価格", inst.format_price(sl)))
            st.markdown(_estimate_html(rows), unsafe_allow_html=True)
            if free is not None and margin > free:
                st.warning("余力が足りません。数量を減らすか、レバレッジを上げてください。")
            if state.get("order_date"):
                st.caption("約定日を入れた場合、取引金額はその日の終値で計算し直します。")

        label = "ロング（買い）で発注" if state["order_side"] == LONG else "ショート（売り）で発注"
        st.button(label, type="primary", on_click=_place_order, disabled=quote is None, width="stretch")


def _positions_section(sim: Simulator, portfolio: Portfolio, today: date) -> None:
    st.subheader("保有中の建玉")
    if not portfolio.positions:
        st.info("保有中の建玉はありません。チャート右の注文パネルから建てられます。")
        return
    prices, _ = sim.latest_prices({p.symbol for p in portfolio.positions})
    _show_table(
        _positions_frame(portfolio, prices),
        pnl_columns=("評価損益",),
        right_columns=("数量", "レバレッジ", "建値", "現在値", "必要証拠金"),
    )
    labels = {}
    for p in portfolio.positions:
        inst = get_instrument(p.symbol)
        labels[p.id] = (
            f"#{p.id} {inst.name} {SIDE_LABELS[p.side]} {inst.format_qty(p.qty)}"
            f" @ {inst.format_price(p.entry_price)}（{format_leverage(p.leverage)}）"
        )
    state = st.session_state
    st.selectbox("建玉を選ぶ", list(labels), format_func=labels.get, key="close_id")
    selected = next(p for p in portfolio.positions if p.id == state["close_id"])
    # 選んだ建玉が変わったら、その建玉の決済注文を入力欄に読み込む
    if state.get("exit_loaded_for") != selected.id:
        state["exit_tp"], state["exit_sl"] = _exit_amounts(selected)
        state["exit_loaded_for"] = selected.id

    close_column, exit_column = st.columns(2, gap="large")
    with close_column:
        st.markdown("**決済する**")
        c1, c2 = st.columns(2)
        c1.number_input("数量（0 ならすべて）", min_value=0, step=1, key="close_qty")
        c2.date_input("決済日（空欄なら最新の価格）", value=None, min_value=OLDEST_DATE, max_value=today, key="close_date")
        with st.container(horizontal=True):
            st.button("決済する", type="primary", on_click=_close_selected, icon=":material/check:")
            with st.popover("すべて決済", icon=":material/done_all:"):
                st.write(f"保有中の建玉 {len(portfolio.positions)} 件をすべて決済します。")
                st.button("すべて決済する", type="primary", on_click=_close_all, key="close_all_confirm")
    with exit_column:
        st.markdown("**決済注文（利益確定・損切り）**")
        c1, c2 = st.columns(2)
        c1.number_input("利益確定（円）", min_value=0, step=EXIT_STEP, key="exit_tp", help="0 なら注文しません。")
        c2.number_input("損切り（円）", min_value=0, step=EXIT_STEP, key="exit_sl", help="0 なら注文しません。")
        st.button("決済注文を設定", on_click=_set_exit, icon=":material/flag:")
        st.caption("注文を出した後の値動きで判定します。アプリを閉じていた間の値動きも、5分足をさかのぼって判定します。")


def _trade_tab(sim: Simulator, portfolio: Portfolio, status: AccountStatus) -> None:
    _show_flash()
    today = sim.clock().date()
    c1, c2, c3, c4 = st.columns([2, 5, 2, 2], vertical_alignment="bottom")
    symbol = c1.selectbox("銘柄", [i.key for i in INSTRUMENTS], format_func=_name, key="symbol")
    timeframe_key = c2.segmented_control(
        "足",
        [t.key for t in TIMEFRAMES],
        format_func=lambda key: get_timeframe(key).label,
        default="1d",
        key="chart_timeframe",
    )
    timeframe = get_timeframe(timeframe_key or "1d")
    periods = {period: label for label, period in timeframe.periods}
    period = c3.selectbox(
        "期間",
        list(periods),
        format_func=periods.get,
        index=list(periods).index(timeframe.default_period),
        key=f"chart_period_{timeframe.key}",
    )
    show_ma = c4.checkbox("移動平均線", value=True, key="chart_ma")
    inst = get_instrument(symbol)

    chart_column, order_column = st.columns([7, 3], gap="medium")
    with chart_column:
        try:
            candles = sim.source.candles(inst, timeframe, period)
        except PriceError as e:
            st.warning(str(e))
        else:
            figure = candle_figure(inst, candles, timeframe, period, portfolio.trades, portfolio.positions, show_ma=show_ma)
            st.plotly_chart(figure, key="candle_chart", config={"displaylogo": False, "scrollZoom": True})
            st.caption(
                "▲ ロングで建てた位置　▼ ショートで建てた位置　× 決済した位置　点線は保有中の建値、破線は決済注文の価格。"
                " ドラッグで移動、ホイールで拡大できます。"
            )
    with order_column:
        _order_panel(sim, inst, status, today)

    _positions_section(sim, portfolio, today)


# ---- 損益・履歴タブ ---------------------------------------------------------


def _pnl_tab(portfolio: Portfolio, rows: list[SummaryRow], errors: list[str], sim: Simulator) -> None:
    if not portfolio.trades and not portfolio.positions:
        st.info("まだ取引がありません。「取引」タブで取引すると、ここに損益の推移が表示されます。")
        return
    trades = len(portfolio.trades)
    wins = sum(1 for t in portfolio.trades if t.pnl > 0)
    losses = sum(1 for t in portfolio.trades if t.pnl < 0)
    realized = sum(r.realized for r in rows)
    c1, c2, c3 = st.columns(3)
    _kpi(c1, "実現損益", fmt_yen(realized), f"決済 {trades} 回", "決済した取引の損益の合計", _pnl_color_of(realized))
    _kpi(c2, "勝率", fmt_rate(wins / trades if trades else None), f"{wins}勝 {losses}敗", "利益が出た決済の割合")
    counts = {reason: sum(1 for t in portfolio.trades if t.reason == reason) for reason in REASON_LABELS}
    _kpi(
        c3,
        "ロスカット",
        f"{counts[LOSSCUT]} 回",
        "・".join(f"{label} {counts[reason]} 回" for reason, label in REASON_LABELS.items() if reason != LOSSCUT),
        "ロスカットで強制決済された回数と、決済注文（利確・損切り）で決済された回数",
    )

    current_total = None
    if not errors:
        current_total = sum(r.realized for r in rows) + sum(r.unrealized or 0 for r in rows)
    left, right = st.columns(2)
    with left:
        st.subheader("通算損益の推移")
        st.plotly_chart(
            equity_figure(portfolio.trades, current_total, sim.clock()), key="equity_chart", config={"displaylogo": False}
        )
    with right:
        st.subheader("銘柄別の損益")
        st.plotly_chart(pnl_by_instrument_figure(rows), key="pnl_chart", config={"displaylogo": False})
    st.subheader("銘柄別の内訳")
    _show_table(_summary_frame(rows), pnl_columns=("実現損益", "評価損益", "合計"), right_columns=("決済回数", "勝率", "建玉数"))


def _history_tab(portfolio: Portfolio) -> None:
    if not portfolio.trades:
        st.info("決済済みの取引はまだありません。")
        return
    frame = _history_frame(portfolio)
    _show_table(frame, pnl_columns=("損益",), right_columns=("数量", "レバレッジ", "建値", "決済値"))
    st.download_button(
        "CSV で保存",
        frame.to_csv(index=False).encode("utf-8-sig"),  # Excel で文字化けしないよう BOM 付き
        file_name="trade_history.csv",
        mime="text/csv",
        icon=":material/download:",
    )


# ---- 全体 ----------------------------------------------------------------


def _dashboard() -> None:
    sim = _simulator()
    try:
        events = sim.check_triggers()
        portfolio = sim.load()
        status, errors = sim.account_status()
        rows, _ = sim.summary_rows()
    except USER_ERRORS as e:
        st.error(str(e))
        return
    if events is not None:
        st.session_state["notice"] = {"losscut": events.losscut, "text": "\n".join(events.lines)}
    notice = st.session_state.get("notice")
    if notice:
        text = notice["text"].replace("\n", "  \n")
        if notice["losscut"]:
            st.error(text, icon=":material/warning:")
        else:
            st.info(text, icon=":material/notifications:")

    st.caption(f"最終更新 {sim.clock():%Y-%m-%d %H:%M:%S}（日本時間）　データ: {sim.path}")
    _account_header(status, errors)
    _quotes(sim)
    trade, pnl, history = st.tabs(["取引", "損益", "取引履歴"], key="main_tabs")
    with trade:
        _trade_tab(sim, portfolio, status)
    with pnl:
        _pnl_tab(portfolio, rows, errors, sim)
    with history:
        _history_tab(portfolio)


def _sidebar() -> None:
    state = st.session_state
    state.setdefault("data_file", DEFAULT_FILE)
    state.setdefault("refresh_interval", "1分")
    with st.sidebar:
        st.header("設定")
        st.text_input("取引データの保存先", key="data_file", help="同じファイルを使えば、コマンドライン版とデータを共有できます。")
        st.toggle("価格を自動で更新する", key="auto_refresh")
        st.selectbox("更新の間隔", list(REFRESH_CHOICES), key="refresh_interval", disabled=not state.get("auto_refresh"))
        st.button("価格を今すぐ更新", on_click=_refresh_prices, icon=":material/refresh:", width="stretch")

        st.subheader("口座")
        try:
            portfolio = _simulator().load()
        except USER_ERRORS as e:
            st.error(str(e))
        else:
            # データファイルを切り替えたら、そのファイルの設定を読み込み直す
            if state.get("account_loaded_from") != state["data_file"]:
                state["account_capital"] = int(portfolio.initial_capital)
                state["account_losscut"] = float(portfolio.losscut_level)
                state["account_loaded_from"] = state["data_file"]
            st.number_input(
                "初期資金（円）", min_value=100_000, step=1_000_000, key="account_capital", on_change=_save_account
            )
            st.number_input(
                "ロスカット水準（証拠金維持率 %）",
                min_value=0.0,
                max_value=100.0,
                step=10.0,
                key="account_losscut",
                on_change=_save_account,
                help="証拠金維持率がこれを下回ると、すべての建玉を決済します。0 にするとロスカットしません。",
            )
        st.divider()
        st.caption(
            "価格は Yahoo Finance のデータです（東証の銘柄はおよそ 20 分遅れ）。"
            "手数料、スプレッド、スワップポイント、信用取引の金利・貸株料は含みません。"
            "ロスカットは、アプリで価格を確認したとき（表示・注文・自動更新）に判定します。"
        )


def main() -> None:
    st.set_page_config(
        page_title="取引シミュレーター",
        page_icon=":material/candlestick_chart:",
        layout="wide",
        initial_sidebar_state="collapsed",
    )
    _sidebar()
    st.title("株・為替 取引シミュレーター")
    interval = REFRESH_CHOICES[st.session_state["refresh_interval"]] if st.session_state.get("auto_refresh") else None
    st.fragment(_dashboard, run_every=interval)()
''',
}

_package_dir = Path.cwd() / "trade_sim_app"
for _name, _text in _SOURCES.items():
    _path = _package_dir / _name
    _path.parent.mkdir(parents=True, exist_ok=True)
    _path.write_text(_text, encoding="utf-8")
if str(_package_dir) not in sys.path:
    sys.path.insert(0, str(_package_dir))
for _module in [m for m in sys.modules if m == "trade_sim" or m.startswith("trade_sim.")]:
    del sys.modules[_module]  # セルを実行し直したときに、書き出したコードを読み込み直す

# 取引データの保存先
DATA_FILE = str(Path.cwd() / "portfolio.json")
if save_to_drive:
    try:
        from google.colab import drive
    except ImportError:  # Colab 以外 (手元の Jupyter など) ではカレントディレクトリに保存する
        pass
    else:
        drive.mount("/content/drive")
        DATA_FILE = "/content/drive/MyDrive/trade_sim/portfolio.json"
else:
    print("※ 取引データはこのランタイムの中だけに保存されます (ランタイムが終了すると消えます)。")

print("準備ができました。取引データの保存先:", DATA_FILE)

In [ ]:
#@title ② 取引画面を表示 { display-mode: "form" }
#@markdown 下に表示される画面で注文・決済ができます。「価格を更新」を押すと最新の価格を取り直します。
from trade_sim.notebook import show

_app = show(DATA_FILE)

In [ ]:
#@title （必要なときだけ）取引データをすべて削除 { display-mode: "form" }
#@markdown 建玉と取引履歴をすべて消して、最初からやり直します。**delete_all** にチェックを入れてから、このセルを実行してください。
delete_all = False  #@param {type:"boolean"}

from trade_sim.simulator import Simulator

if delete_all:
    print(*Simulator(DATA_FILE).reset().lines)
else:
    print("削除するときは delete_all にチェックを入れてから、このセルを実行してください。")